# Process Polymarket Data After LLM Parsing

This notebook cleans and organizes Polymarket stock-price market data for later option-pricing and Bloomberg data collection.

## Notebook structure

1. Setup and load data
2. Parse and verify timestamps
3. Clean market bounds
4. Classify interval types and summarize
5. Check event-market consistency
6. Filter to usable events for Bloomberg daily option history
7. Build event-level and strike-level request tables for Bloomberg

Throughout the notebook:

- `events_data` refers to the event-level table
- `markets_data` refers to the market-level table
- the authoritative stock ticker for pricing and Bloomberg work will come from `markets_data["ticker"]`

In [1]:
import pandas as pd
import numpy as np
import re

ET_TZ = "America/New_York"

EVENTS_FILE = "stocks_events_close_only.csv"
MARKETS_FILE = "stocks_markets_close_only.csv"

events_data = pd.read_csv(EVENTS_FILE)
markets_data = pd.read_csv(MARKETS_FILE)

print("Raw shapes:")
print("  events_data :", events_data.shape)
print("  markets_data:", markets_data.shape)

events_data.head(3), markets_data.head(3)

Raw shapes:
  events_data : (624, 19)
  markets_data: (5918, 17)


(      id                       ticker                         slug  \
 0  72220  aapl-above-in-november-2025  aapl-above-in-november-2025   
 1  72221  msft-above-in-november-2025  msft-above-in-november-2025   
 2  72222  amzn-above-in-november-2025  amzn-above-in-november-2025   
 
                                                title  \
 0  Will Apple (AAPL) close above ___ end of Novem...   
 1  Will Microsoft (MSFT) close above ___ end of N...   
 2  Will Amazon (AMZN) close above ___ end of Nove...   
 
                                          description category  \
 0  This market will resolve to "Yes" if the offic...   Stocks   
 1  This market will resolve to "Yes" if the offic...   Stocks   
 2  This market will resolve to "Yes" if the offic...   Stocks   
 
                    startDate               creationDate  \
 0  2025-11-01 04:00:00+00:00  2025-11-01 04:00:00+00:00   
 1  2025-11-01 04:00:00+00:00  2025-11-01 04:00:00+00:00   
 2  2025-11-01 04:00:00+00:00  2025-11

## Parse timestamps

The exported CSV files store timestamps as strings.  
We parse them back into timezone-aware datetimes.

For the event file, the UTC columns are:

- `startDate`
- `creationDate`
- `endDate`

and the saved ET columns are:

- `startDate_et`
- `creationDate_et`
- `endDate_et`

For the market file, the UTC column is:

- `endDate`

and the saved ET column is:

- `endDate_et`

We parse all timestamp strings with `utc=True`, then convert the saved ET columns back to `America/New_York`. This avoids pandas mixed-timezone warnings and preserves daylight-saving-time transitions correctly.

In [2]:
event_utc_cols = ["startDate", "creationDate", "endDate"]
event_et_cols = ["startDate_et", "creationDate_et", "endDate_et"]

market_utc_cols = ["endDate"]
market_et_cols = ["endDate_et"]

def parse_utc_columns(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(
                df[c].astype("string").str.strip(),
                format="mixed",
                errors="coerce",
                utc=True
            )
    return df

def parse_saved_et_columns(df, cols, tz_name=ET_TZ):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(
                df[c].astype("string").str.strip(),
                format="mixed",
                errors="coerce",
                utc=True
            ).dt.tz_convert(tz_name)
    return df

events_data = parse_utc_columns(events_data, event_utc_cols)
events_data = parse_saved_et_columns(events_data, event_et_cols)

markets_data = parse_utc_columns(markets_data, market_utc_cols)
markets_data = parse_saved_et_columns(markets_data, market_et_cols)

## Verify UTC → ET conversion

We recompute the ET columns from the UTC columns and verify that they exactly match the saved ET columns.

This confirms that:

1. the parsed timestamps are timezone-aware,
2. the UTC and ET columns refer to the same underlying instants,
3. daylight saving time is handled correctly.

In [3]:
events_data["startDate_et_check"] = events_data["startDate"].dt.tz_convert(ET_TZ)
events_data["creationDate_et_check"] = events_data["creationDate"].dt.tz_convert(ET_TZ)
events_data["endDate_et_check"] = events_data["endDate"].dt.tz_convert(ET_TZ)

markets_data["endDate_et_check"] = markets_data["endDate"].dt.tz_convert(ET_TZ)

print("Event ET conversion checks:")
print("  startDate_et exact match   :", (events_data["startDate_et"] == events_data["startDate_et_check"]).mean())
print("  creationDate_et exact match:", (events_data["creationDate_et"] == events_data["creationDate_et_check"]).mean())
print("  endDate_et exact match     :", (events_data["endDate_et"] == events_data["endDate_et_check"]).mean())

print("\nMarket ET conversion checks:")
print("  endDate_et exact match     :", (markets_data["endDate_et"] == markets_data["endDate_et_check"]).mean())

Event ET conversion checks:
  startDate_et exact match   : 1.0
  creationDate_et exact match: 1.0
  endDate_et exact match     : 1.0

Market ET conversion checks:
  endDate_et exact match     : 1.0


## Quick inspection

We display a few rows to visually confirm that the timestamps look correct.

For example, a UTC timestamp such as `2025-11-28 21:00:00+00:00` should convert to `2025-11-28 16:00:00-05:00`, which aligns with the U.S. stock-market close in New York time.

In [4]:
events_data.loc[:4, [
    "id",
    "startDate", "startDate_et", "startDate_et_check",
    "endDate", "endDate_et", "endDate_et_check"
]]

,id,startDate,startDate_et,startDate_et_check,endDate,endDate_et,endDate_et_check
0,72220,2025-11-01 04:00:00+00:00,2025-11-01 00:00:00-04:00,2025-11-01 00:00:00-04:00,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00
1,72221,2025-11-01 04:00:00+00:00,2025-11-01 00:00:00-04:00,2025-11-01 00:00:00-04:00,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00
2,72222,2025-11-01 04:00:00+00:00,2025-11-01 00:00:00-04:00,2025-11-01 00:00:00-04:00,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00
3,72223,2025-11-01 04:00:00+00:00,2025-11-01 00:00:00-04:00,2025-11-01 00:00:00-04:00,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00
4,72224,2025-11-01 04:00:00+00:00,2025-11-01 00:00:00-04:00,2025-11-01 00:00:00-04:00,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00


In [5]:
markets_data.loc[:4, [
    "id", "event_id",
    "endDate", "endDate_et", "endDate_et_check"
]]

,id,event_id,endDate,endDate_et,endDate_et_check
0,662936,72220,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00
1,662937,72220,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00
2,662938,72220,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00
3,662939,72220,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00
4,662940,72220,2025-11-28 21:00:00+00:00,2025-11-28 16:00:00-05:00,2025-11-28 16:00:00-05:00


## Clean weird market bounds

The market-level table contains a small number of malformed or economically trivial payoff bounds after LLM parsing.

We first flag "weird" rows using the following criteria:

- `lower_bound <= 0`
- `lower_bound is NaN`
- `upper_bound <= 0`

We then apply three deterministic rules:

### Rule 1: lower-tail markets
If `lower_bound` is missing and the question clearly describes a lower-tail event
(for example, `below`, `under`, or `<`) with a positive upper threshold `X`,
we set the interval to `[0, X]`.

### Rule 2: trivial or degenerate markets
If `lower_bound` is `0` or missing and `upper_bound` is `0` or `inf`,
we drop the row as trivial or degenerate.

Examples include:
- `close above $0`
- `close at <$0`

### Rule 3: malformed upper-tail markets
If `lower_bound` is missing or clearly invalid and the question clearly describes
an upper-tail event (for example, `above` or `>`) with threshold `X`,
we fix the interval to `[X, inf)`.

### Manual review
Any weird row not handled by the three rules is left unresolved for manual inspection.

We save:
- the cleaned market dataset
- the initial weird rows
- the rows fixed by Rule 1
- the rows fixed by Rule 3
- the rows dropped by Rule 2
- the unresolved weird rows

In [6]:
def clean_market_bounds(df, invalid_lower_cutoff=-1e9, tol=1e-6, verbose=True):
    """
    Clean market bounds with three deterministic rules.

    Weird rows are defined as:
      - lower_bound <= 0
      - lower_bound is NaN
      - upper_bound <= 0

    Rule 1:
      lower_bound is NaN and question is lower-tail (below / under / <)
      with upper_bound = X > 0
      -> set lower_bound = 0

    Rule 2:
      lower_bound == 0 or NaN and upper_bound == 0 or inf
      -> drop as trivial / degenerate

    Rule 3:
      lower_bound is NaN or invalid large negative, and question is upper-tail
      (above / >) with upper_bound = X
      -> set lower_bound = X, upper_bound = inf
    """
    out = df.copy()

    out["lower_bound"] = pd.to_numeric(out["lower_bound"], errors="coerce")
    out["upper_bound"] = pd.to_numeric(out["upper_bound"], errors="coerce")

    # --------------------------------------------------
    # Step 1: identify weird rows
    # --------------------------------------------------
    out["flag_lower_nan"] = out["lower_bound"].isna()
    out["flag_lower_le_0"] = out["lower_bound"].le(0).fillna(False)
    out["flag_upper_le_0"] = out["upper_bound"].le(0).fillna(False)

    out["flag_any_weird"] = (
        out["flag_lower_nan"] |
        out["flag_lower_le_0"] |
        out["flag_upper_le_0"]
    )

    weird_rows_initial = out.loc[out["flag_any_weird"]].copy()

    # Save original values for reporting
    out["lower_bound_before"] = out["lower_bound"]
    out["upper_bound_before"] = out["upper_bound"]

    # --------------------------------------------------
    # Extract thresholds from question text
    # --------------------------------------------------
    def extract_above_threshold(q):
        if pd.isna(q):
            return np.nan
        q = str(q).lower()
        patterns = [
            r"(?:close|finish|end|closes)\s+above\s+\$?\s*([0-9]+(?:\.[0-9]+)?)",
            r"(?:close|finish|end|closes)\s*>\s*\$?\s*([0-9]+(?:\.[0-9]+)?)",
            r"above\s+\$?\s*([0-9]+(?:\.[0-9]+)?)",
            r">\s*\$?\s*([0-9]+(?:\.[0-9]+)?)",
        ]
        for pat in patterns:
            m = re.search(pat, q)
            if m:
                return float(m.group(1))
        return np.nan

    def extract_below_threshold(q):
        if pd.isna(q):
            return np.nan
        q = str(q).lower()
        patterns = [
            r"(?:close|finish|end|closes)\s+(?:below|under)\s+\$?\s*([0-9]+(?:\.[0-9]+)?)",
            r"(?:close|finish|end|closes)\s*<\s*\$?\s*([0-9]+(?:\.[0-9]+)?)",
            r"(?:below|under)\s+\$?\s*([0-9]+(?:\.[0-9]+)?)",
            r"<\s*\$?\s*([0-9]+(?:\.[0-9]+)?)",
            r"at or below\s+\$?\s*([0-9]+(?:\.[0-9]+)?)",
        ]
        for pat in patterns:
            m = re.search(pat, q)
            if m:
                return float(m.group(1))
        return np.nan

    out["threshold_above_q"] = out["question"].apply(extract_above_threshold)
    out["threshold_below_q"] = out["question"].apply(extract_below_threshold)

    # --------------------------------------------------
    # Rule 1: lower-tail market -> [0, X]
    # --------------------------------------------------
    mask_rule1 = (
        out["lower_bound"].isna() &
        np.isfinite(out["upper_bound"]) &
        (out["upper_bound"] > 0) &
        np.isfinite(out["threshold_below_q"]) &
        ((out["upper_bound"] - out["threshold_below_q"]).abs() < tol)
    )

    out.loc[mask_rule1, "lower_bound"] = 0.0

    fixed_rule1_df = (
        out.loc[mask_rule1, [
            c for c in [
                "id", "question", "category", "event_id", "event_slug", "event_title",
                "ticker",
                "lower_bound_before", "upper_bound_before",
                "lower_bound", "upper_bound",
                "threshold_below_q"
            ] if c in out.columns
        ]]
        .copy()
        .rename(columns={
            "lower_bound": "lower_bound_after",
            "upper_bound": "upper_bound_after"
        })
    )
    fixed_rule1_df["fix_type"] = "rule1_nan_lower_to_zero_for_lower_tail"

    # --------------------------------------------------
    # Rule 3: malformed upper-tail market -> [X, inf)
    # --------------------------------------------------
    mask_rule3 = (
        (out["lower_bound"].isna() | (out["lower_bound"] < invalid_lower_cutoff)) &
        np.isfinite(out["upper_bound"]) &
        np.isfinite(out["threshold_above_q"]) &
        ((out["upper_bound"] - out["threshold_above_q"]).abs() < tol)
    )

    out.loc[mask_rule3, "lower_bound"] = out.loc[mask_rule3, "upper_bound"]
    out.loc[mask_rule3, "upper_bound"] = np.inf

    fixed_rule3_df = (
        out.loc[mask_rule3, [
            c for c in [
                "id", "question", "category", "event_id", "event_slug", "event_title",
                "ticker",
                "lower_bound_before", "upper_bound_before",
                "lower_bound", "upper_bound",
                "threshold_above_q"
            ] if c in out.columns
        ]]
        .copy()
        .rename(columns={
            "lower_bound": "lower_bound_after",
            "upper_bound": "upper_bound_after"
        })
    )
    fixed_rule3_df["fix_type"] = "rule3_bad_lower_to_upper_tail_interval"

    # --------------------------------------------------
    # Rule 2: trivial / degenerate rows -> drop
    # --------------------------------------------------
    lower_is_zero_or_nan = out["lower_bound"].isna() | (out["lower_bound"] == 0)
    upper_is_zero_or_inf = (out["upper_bound"] == 0) | np.isinf(out["upper_bound"])

    mask_rule2_drop = lower_is_zero_or_nan & upper_is_zero_or_inf

    dropped_rule2_df = (
        out.loc[mask_rule2_drop, [
            c for c in [
                "id", "question", "category", "event_id", "event_slug", "event_title",
                "ticker",
                "lower_bound_before", "upper_bound_before",
                "lower_bound", "upper_bound"
            ] if c in out.columns
        ]]
        .copy()
        .rename(columns={
            "lower_bound": "lower_bound_after",
            "upper_bound": "upper_bound_after"
        })
    )
    dropped_rule2_df["drop_type"] = "rule2_drop_trivial_or_degenerate"

    # --------------------------------------------------
    # Weird but unresolved
    # --------------------------------------------------
    was_fixed_or_dropped = mask_rule1 | mask_rule3 | mask_rule2_drop

    weird_unresolved_df = (
        out.loc[
            out["flag_any_weird"] & (~was_fixed_or_dropped),
            [
                c for c in [
                    "id", "question", "category", "event_id", "event_slug", "event_title",
                    "ticker",
                    "lower_bound_before", "upper_bound_before",
                    "lower_bound", "upper_bound",
                    "flag_lower_nan", "flag_lower_le_0", "flag_upper_le_0",
                    "threshold_below_q", "threshold_above_q"
                ] if c in out.columns
            ]
        ]
        .copy()
        .rename(columns={
            "lower_bound": "lower_bound_after",
            "upper_bound": "upper_bound_after"
        })
    )

    # --------------------------------------------------
    # Final cleaned dataframe
    # --------------------------------------------------
    cleaned_df = out.loc[~mask_rule2_drop].copy()

    drop_helper_cols = [
        "flag_lower_nan", "flag_lower_le_0", "flag_upper_le_0", "flag_any_weird",
        "lower_bound_before", "upper_bound_before",
        "threshold_above_q", "threshold_below_q"
    ]
    cleaned_df = cleaned_df.drop(columns=[c for c in drop_helper_cols if c in cleaned_df.columns])

    if verbose:
        print(f"Initial weird rows: {len(weird_rows_initial)}")
        print(f"Fixed by rule 1 : {len(fixed_rule1_df)}")
        print(f"Fixed by rule 3 : {len(fixed_rule3_df)}")
        print(f"Dropped by rule 2: {len(dropped_rule2_df)}")
        print(f"Unresolved weird : {len(weird_unresolved_df)}")

    return (
        cleaned_df,
        weird_rows_initial,
        fixed_rule1_df,
        fixed_rule3_df,
        dropped_rule2_df,
        weird_unresolved_df
    )

In [7]:
(
    markets_data_cleaned,
    weird_rows_initial,
    fixed_rule1_df,
    fixed_rule3_df,
    dropped_rule2_df,
    weird_unresolved_df
) = clean_market_bounds(markets_data, verbose=True)

Initial weird rows: 207
Fixed by rule 1 : 174
Fixed by rule 3 : 18
Dropped by rule 2: 11
Unresolved weird : 4


## Inspect cleaning outputs

We inspect the four outputs:

- initial weird rows
- rows fixed by Rule 1
- rows fixed by Rule 3
- rows dropped by Rule 2
- weird rows still unresolved

The unresolved rows should be a very small set that requires manual review.

In [8]:
fixed_rule1_df.head(10)

,id,question,category,event_id,event_slug,event_title,ticker,lower_bound_before,upper_bound_before,lower_bound_after,upper_bound_after,threshold_below_q,fix_type
312,677433,Will Apple (AAPL) close at <$245 on the final ...,Stocks,79247,aapl-week-november-21-2025,Apple (AAPL) closes week of Nov 17 at ___?,AAPL,NaN,245.0,0.0,245.0,245.0,rule1_nan_lower_to_zero_for_lower_tail
328,677444,Will Microsoft (MSFT) close at <$460 on the fi...,Stocks,79248,msft-week-november-21-2025,Microsoft (MSFT) closes week of Nov 17 at ___?,MSFT,NaN,460.0,0.0,460.0,460.0,rule1_nan_lower_to_zero_for_lower_tail
339,677455,Will Amazon (AMZN) close at <$225 on the final...,Stocks,79249,amzn-week-november-21-2025,Amazon (AMZN) closes week of Nov 17 at ___?,AMZN,NaN,225.0,0.0,225.0,225.0,rule1_nan_lower_to_zero_for_lower_tail
345,677466,Will Google (GOOGL) close at <$270 on the fina...,Stocks,79250,googl-week-november-21-2025,Google (GOOGL) closes week of Nov 17 at ___?,GOOGL,NaN,270.0,0.0,270.0,270.0,rule1_nan_lower_to_zero_for_lower_tail
356,677477,Will Meta (META) close at <$590 on the final d...,Stocks,79251,meta-week-november-21-2025,Meta (META) closes week of Nov 17 at ___?,META,NaN,590.0,0.0,590.0,590.0,rule1_nan_lower_to_zero_for_lower_tail
372,677488,Will Tesla (TSLA) close at <$425 on the final ...,Stocks,79252,tsla-week-november-21-2025,Tesla (TSLA) closes week of Nov 17 at ___?,TSLA,NaN,425.0,0.0,425.0,425.0,rule1_nan_lower_to_zero_for_lower_tail
383,677499,Will NVIDIA (NVDA) close at <$175 on the final...,Stocks,79253,nvda-week-november-21-2025,NVIDIA (NVDA) closes week of Nov 17 at ___?,NVDA,NaN,175.0,0.0,175.0,175.0,rule1_nan_lower_to_zero_for_lower_tail
389,677510,Will Palantir (PLTR) close at <$184 on the fin...,Stocks,79254,pltr-week-november-21-2025,Palantir (PLTR) closes week of Nov 17 at ___?,PLTR,NaN,184.0,0.0,184.0,184.0,rule1_nan_lower_to_zero_for_lower_tail
400,677521,Will Opendoor (OPEN) close at <$3.00 on the fi...,Stocks,79255,open-week-november-21-2025,Opendoor (OPEN) closes week of Nov 17 at ___?,OPEN,NaN,3.0,0.0,3.0,3.0,rule1_nan_lower_to_zero_for_lower_tail
416,677532,Will Rocket Lab (RKLB) close at <$42 on the fi...,Stocks,79256,rklb-week-november-21-2025,Rocket Lab (RKLB) closes week of Nov 17 at ___?,RKLB,NaN,42.0,0.0,42.0,42.0,rule1_nan_lower_to_zero_for_lower_tail


In [9]:
fixed_rule1_df.tail(10)

,id,question,category,event_id,event_slug,event_title,ticker,lower_bound_before,upper_bound_before,lower_bound_after,upper_bound_after,threshold_below_q,fix_type
5585,1666183,Will Opendoor (OPEN) close at <$1.00 on the fi...,Stocks,290952,open-week-march-27-2026,Opendoor (OPEN) closes week of Mar 23 at ___?,OPEN,NaN,1.0,0.0,1.0,1.0,rule1_nan_lower_to_zero_for_lower_tail
5662,1747931,Will Apple (AAPL) close at <$230 on the final ...,Stocks,315442,aapl-week-april-3-2026,Apple (AAPL) closes week of Mar 30 at ___?,AAPL,NaN,230.0,0.0,230.0,230.0,rule1_nan_lower_to_zero_for_lower_tail
5673,1747992,Will Microsoft (MSFT) close at <$320 on the fi...,Stocks,315456,msft-week-april-3-2026,Microsoft (MSFT) closes week of Mar 30 at ___?,MSFT,NaN,320.0,0.0,320.0,320.0,rule1_nan_lower_to_zero_for_lower_tail
5697,1748044,Will Amazon (AMZN) close at <$185 on the final...,Stocks,315461,amzn-week-april-3-2026,Amazon (AMZN) closes week of Mar 30 at ___?,AMZN,NaN,185.0,0.0,185.0,185.0,rule1_nan_lower_to_zero_for_lower_tail
5725,1748097,Will Google (GOOGL) close at <$260 on the fina...,Stocks,315467,googl-week-april-3-2026,Google (GOOGL) closes week of Mar 30 at ___?,GOOGL,NaN,260.0,0.0,260.0,260.0,rule1_nan_lower_to_zero_for_lower_tail
5745,1748143,Will Meta (META) close at <$500 on the final d...,Stocks,315472,meta-week-april-3-2026,Meta (META) closes week of Mar 30 at ___?,META,NaN,500.0,0.0,500.0,500.0,rule1_nan_lower_to_zero_for_lower_tail
5767,1748187,Will Tesla (TSLA) close at <$350 on the final ...,Stocks,315477,tsla-week-april-3-2026,Tesla (TSLA) closes week of Mar 30 at ___?,TSLA,NaN,350.0,0.0,350.0,350.0,rule1_nan_lower_to_zero_for_lower_tail
5798,1748238,Will NVIDIA (NVDA) close at <$150 on the final...,Stocks,315480,nvda-week-april-3-2026,NVIDIA (NVDA) closes week of Mar 30 at ___?,NVDA,NaN,150.0,0.0,150.0,150.0,rule1_nan_lower_to_zero_for_lower_tail
5820,1748289,Will Netflix (NFLX) close at <$50 on the final...,Stocks,315484,nflx-week-april-3-2026,Netflix (NFLX) closes week of Mar 30 at ___?,NFLX,NaN,50.0,0.0,50.0,50.0,rule1_nan_lower_to_zero_for_lower_tail
5839,1748339,Will Palantir (PLTR) close at <$138 on the fin...,Stocks,315488,pltr-week-april-3-2026,Palantir (PLTR) closes week of Mar 30 at ___?,PLTR,NaN,138.0,0.0,138.0,138.0,rule1_nan_lower_to_zero_for_lower_tail


In [10]:
fixed_rule3_df

,id,question,category,event_id,event_slug,event_title,ticker,lower_bound_before,upper_bound_before,lower_bound_after,upper_bound_after,threshold_above_q,fix_type
3520,1301616,Will Microsoft (MSFT) close above $345 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,345.0,345.0,inf,345.0,rule3_bad_lower_to_upper_tail_interval
3521,1301618,Will Microsoft (MSFT) close above $360 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,360.0,360.0,inf,360.0,rule3_bad_lower_to_upper_tail_interval
3522,1301622,Will Microsoft (MSFT) close above $375 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,375.0,375.0,inf,375.0,rule3_bad_lower_to_upper_tail_interval
3523,1301626,Will Microsoft (MSFT) close above $390 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,390.0,390.0,inf,390.0,rule3_bad_lower_to_upper_tail_interval
3524,1301632,Will Microsoft (MSFT) close above $405 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,405.0,405.0,inf,405.0,rule3_bad_lower_to_upper_tail_interval
3525,1301642,Will Microsoft (MSFT) close above $435 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,435.0,435.0,inf,435.0,rule3_bad_lower_to_upper_tail_interval
3526,1301637,Will Microsoft (MSFT) close above $420 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,420.0,420.0,inf,420.0,rule3_bad_lower_to_upper_tail_interval
3527,1301646,Will Microsoft (MSFT) close above $450 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,450.0,450.0,inf,450.0,rule3_bad_lower_to_upper_tail_interval
3528,1301653,Will Microsoft (MSFT) close above $480 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,480.0,480.0,inf,480.0,rule3_bad_lower_to_upper_tail_interval
3529,1301656,Will Microsoft (MSFT) close above $495 end of ...,Stocks,193936,msft-above-in-february-2026,Will Microsoft (MSFT) close above ___ end of F...,MSFT,-2.147484e+09,495.0,495.0,inf,495.0,rule3_bad_lower_to_upper_tail_interval


In [11]:
dropped_rule2_df

,id,question,category,event_id,event_slug,event_title,ticker,lower_bound_before,upper_bound_before,lower_bound_after,upper_bound_after,drop_type
1175,740915,Will Netflix (NFLX) close above $0.00 end of D...,Stocks,92888,nflx-above-in-december-2025,Will Netflix (NFLX) close above ___ end of Dec...,NFLX,0.0,inf,0.0,inf,rule2_drop_trivial_or_degenerate
2218,1031786,Will Netflix (NFLX) close above $0.00 end of J...,Stocks,125821,nflx-above-in-january-2026,Will Netflix (NFLX) close above ___ end of Jan...,NFLX,0.0,inf,0.0,inf,rule2_drop_trivial_or_degenerate
2291,1031856,Will Opendoor (OPEN) close above $0.00 end of ...,Stocks,125827,open-above-in-january-2026,Will Opendoor (OPEN) close above ___ end of Ja...,OPEN,0.0,inf,0.0,inf,rule2_drop_trivial_or_degenerate
3761,1301908,Will Netflix (NFLX) close above $0.00 end of F...,Stocks,193985,nflx-above-in-february-2026,Will Netflix (NFLX) close above ___ end of Feb...,NFLX,0.0,inf,0.0,inf,rule2_drop_trivial_or_degenerate
3798,1301969,Will Opendoor (OPEN) close above $0.00 end of ...,Stocks,193994,open-above-in-february-2026,Will Opendoor (OPEN) close above ___ end of Fe...,OPEN,0.0,inf,0.0,inf,rule2_drop_trivial_or_degenerate
4202,1349416,Will Opendoor (OPEN) close at <$0 on the final...,Stocks,200820,open-week-february-13-2026,Opendoor (OPEN) closes week of Feb 9 at ___?,OPEN,NaN,0.0,NaN,0.0,rule2_drop_trivial_or_degenerate
4625,1374794,Will Opendoor (OPEN) close at <$0 on the final...,Stocks,208315,open-week-february-20-2026,Opendoor (OPEN) closes week of Feb 16 at ___?,OPEN,NaN,0.0,NaN,0.0,rule2_drop_trivial_or_degenerate
4906,1404047,Will Opendoor (OPEN) close at <$0 on the final...,Stocks,219959,open-week-february-27-2026,Opendoor (OPEN) closes week of Feb 23 at ___?,OPEN,NaN,0.0,NaN,0.0,rule2_drop_trivial_or_degenerate
5118,1460534,Will Netflix (NFLX) close above $0.00 end of M...,Stocks,235635,nflx-above-in-march-2026,Will Netflix (NFLX) close above ___ end of March?,NFLX,0.0,inf,0.0,inf,rule2_drop_trivial_or_degenerate
5148,1460492,Will Opendoor (OPEN) close above $0.00 end of ...,Stocks,235638,open-above-in-march-2026,Will Opendoor (OPEN) close above ___ end of Ma...,OPEN,0.0,inf,0.0,inf,rule2_drop_trivial_or_degenerate


In [12]:
weird_unresolved_df

,id,question,category,event_id,event_slug,event_title,ticker,lower_bound_before,upper_bound_before,lower_bound_after,upper_bound_after,flag_lower_nan,flag_lower_le_0,flag_upper_le_0,threshold_below_q,threshold_above_q
4203,1349417,Will Opendoor (OPEN) close at $0-$1.00 on the ...,Stocks,200820,open-week-february-13-2026,Opendoor (OPEN) closes week of Feb 9 at ___?,OPEN,0.0,1.0,0.0,1.0,False,True,False,NaN,NaN
4626,1374795,Will Opendoor (OPEN) close at $0-$1.00 on the ...,Stocks,208315,open-week-february-20-2026,Opendoor (OPEN) closes week of Feb 16 at ___?,OPEN,0.0,1.0,0.0,1.0,False,True,False,NaN,NaN
4907,1404048,Will Opendoor (OPEN) close at $0-$1.00 on the ...,Stocks,219959,open-week-february-27-2026,Opendoor (OPEN) closes week of Feb 23 at ___?,OPEN,0.0,1.0,0.0,1.0,False,True,False,NaN,NaN
5863,1748391,Will Opendoor (OPEN) close at $0-$1.00 on the ...,Stocks,315492,open-week-april-3-2026,Opendoor (OPEN) closes week of Mar 30 at ___?,OPEN,0.0,1.0,0.0,1.0,False,True,False,NaN,NaN


### Note on the weird_unresolved markets
- After manual confirmation, they can be kept as what they currently are.

## Save cleaned market data and audit tables

We save the cleaned market table and all intermediate audit outputs so that
later sections can use the cleaned dataset directly, without repeating the
cleaning logic.

In [13]:
TEMP_EVENT_COLS = [
    "startDate_et_check",
    "creationDate_et_check",
    "endDate_et_check",
]

TEMP_MARKET_COLS = [
    "endDate_et_check",
]

def prepare_events_for_save(df):
    return df.drop(columns=[c for c in TEMP_EVENT_COLS if c in df.columns]).copy()

def prepare_markets_for_save(df):
    return df.drop(columns=[c for c in TEMP_MARKET_COLS if c in df.columns]).copy()

In [14]:
# Save cleaned market data and audit tables
markets_data_cleaned_to_save = prepare_markets_for_save(markets_data_cleaned)
markets_data_cleaned_to_save.to_csv("stocks_markets_close_only_cleaned.csv", index=False)

# Keep audit tables as-is, since their diagnostic columns are useful
weird_rows_initial.to_csv("weird_rows_initial.csv", index=False)
fixed_rule1_df.to_csv("fixed_rule1_df.csv", index=False)
fixed_rule3_df.to_csv("fixed_rule3_df.csv", index=False)
dropped_rule2_df.to_csv("dropped_rule2_df.csv", index=False)
weird_unresolved_df.to_csv("weird_unresolved_df.csv", index=False)

print("Saved:")
print("- stocks_markets_close_only_cleaned.csv")
print("- weird_rows_initial.csv")
print("- fixed_rule1_df.csv")
print("- fixed_rule3_df.csv")
print("- dropped_rule2_df.csv")
print("- weird_unresolved_df.csv")

Saved:
- stocks_markets_close_only_cleaned.csv
- weird_rows_initial.csv
- fixed_rule1_df.csv
- fixed_rule3_df.csv
- dropped_rule2_df.csv
- weird_unresolved_df.csv


In [15]:
# Replace working market dataframe with the cleaned version
markets_data = markets_data_cleaned_to_save.copy()
print("Current markets_data shape:", markets_data.shape)

Current markets_data shape: (5907, 17)


## Classify interval types after cleaning

After cleaning market bounds, we classify each market into one of four payoff-interval types based only on its numeric bounds:

1. **`[x, inf)`**  
   Upper-tail markets, such as “close above $X”.

2. **`[0, x]`**  
   Lower-tail markets, such as “close below $X”.

3. **`[x, y]`**  
   Finite interval markets.

4. **`other`**  
   Any row that does not fit the three standard forms above.

At this stage, the classification uses the cleaned numeric bounds directly.  
It does not revisit the original question text.

In [16]:
def classify_interval_type(row, tol=1e-12):
    lb = row["lower_bound"]
    ub = row["upper_bound"]

    if pd.isna(lb) or pd.isna(ub):
        return "other"

    # [x, inf)
    if np.isfinite(lb) and np.isinf(ub):
        return "[x, inf)"

    # [0, x]
    if np.isfinite(ub) and abs(lb - 0.0) < tol:
        return "[0, x]"

    # [x, y]
    if np.isfinite(lb) and np.isfinite(ub) and lb > 0 and ub > lb:
        return "[x, y]"

    return "other"

In [17]:
markets_data = markets_data.copy()
markets_data["interval_type"] = markets_data.apply(classify_interval_type, axis=1)

interval_summary = (
    markets_data["interval_type"]
    .value_counts(dropna=False)
    .rename_axis("interval_type")
    .reset_index(name="n_markets")
)

interval_summary["share"] = interval_summary["n_markets"] / interval_summary["n_markets"].sum()

## Quick interpretation

This table shows the distribution of cleaned payoff structures across the market dataset.

In this project:

- `[x, inf)` corresponds to upper-tail digital-call style contracts,
- `[0, x]` corresponds to lower-tail digital-put style contracts,
- `[x, y]` corresponds to interval contracts that can be priced as the difference of two upper-tail digital contracts.

If the cleaning is successful, the `other` category should be very small or empty.

In [18]:
print("Interval summary:")
display(interval_summary)

print("\nAny 'other' rows?")
display(
    markets_data.loc[
        markets_data["interval_type"] == "other",
        ["id", "question", "ticker", "lower_bound", "upper_bound"]
    ].head(20)
)

Interval summary:


,interval_type,n_markets,share
0,"[x, inf)",4130,0.699170
1,"[x, y]",1599,0.270696
2,"[0, x]",178,0.030134



Any 'other' rows?


,id,question,ticker,lower_bound,upper_bound


## Save interval-type outputs

We save:

- the cleaned market table with interval labels
- the interval summary table

In [19]:
markets_data_to_save = prepare_markets_for_save(markets_data)
markets_data_to_save.to_csv("stocks_markets_close_only_cleaned_with_interval_type.csv", index=False)

interval_summary.to_csv("market_interval_type_summary.csv", index=False)

print("Saved:")
print("- stocks_markets_close_only_cleaned_with_interval_type.csv")
print("- market_interval_type_summary.csv")

Saved:
- stocks_markets_close_only_cleaned_with_interval_type.csv
- market_interval_type_summary.csv


In [20]:
markets_data = markets_data_to_save.copy()
print("Current markets_data shape:", markets_data.shape)

Current markets_data shape: (5907, 18)


## Check event-market consistency

Before building Bloomberg request tables, we verify two key consistency conditions between the event-level and market-level data:

1. **End timestamp consistency**  
   For each event, all markets under that event should have the same `endDate`, and that timestamp should match the event-level `endDate`.

2. **Stock ticker consistency within event**  
   For each event, all markets under that event should share the same stock ticker in `markets_data["ticker"]`.

This is important because:

- the event-level table is used for timing,
- the market-level table is used for payoff bounds and stock ticker,
- and the authoritative stock ticker for Bloomberg work should come from `markets_data["ticker"]`, not from `events_data["ticker"]`.

### 4.1 Check end timestamp consistency between events and markets

In [21]:
# Event-level end timestamps
event_end = events_data[["id", "slug", "title", "endDate", "endDate_et"]].copy()
event_end = event_end.rename(columns={
    "id": "event_id",
    "endDate": "event_endDate",
    "endDate_et": "event_endDate_et"
})

# Market-level end timestamp summary by event
market_end_summary = (
    markets_data.groupby("event_id")
    .agg(
        n_markets=("id", "size"),
        n_unique_market_endDate=("endDate", "nunique"),
        n_unique_market_endDate_et=("endDate_et", "nunique"),
        market_endDate_first=("endDate", "first"),
        market_endDate_et_first=("endDate_et", "first"),
    )
    .reset_index()
)

# Merge and compare
enddate_check = event_end.merge(market_end_summary, on="event_id", how="left")

enddate_check["utc_match"] = enddate_check["event_endDate"] == enddate_check["market_endDate_first"]
enddate_check["et_match"] = enddate_check["event_endDate_et"] == enddate_check["market_endDate_et_first"]

enddate_summary = {
    "n_events": len(enddate_check),
    "all_markets_same_utc_within_event": (enddate_check["n_unique_market_endDate"] == 1).mean(),
    "all_markets_same_et_within_event": (enddate_check["n_unique_market_endDate_et"] == 1).mean(),
    "event_equals_market_utc": enddate_check["utc_match"].mean(),
    "event_equals_market_et": enddate_check["et_match"].mean(),
}

enddate_summary

{'n_events': 624,
 'all_markets_same_utc_within_event': 1.0,
 'all_markets_same_et_within_event': 1.0,
 'event_equals_market_utc': 1.0,
 'event_equals_market_et': 1.0}

### 4.2 Check stock ticker consistency within each event

The event-level `ticker` column is a Polymarket-side identifier and should not be treated as the authoritative stock ticker for Bloomberg work.

Instead, we verify that all markets under a given event share the same value of `markets_data["ticker"]`.  
If so, we can define an authoritative `stock_ticker` at the event level using the market table.

In [22]:
ticker_check = (
    markets_data.groupby("event_id")
    .agg(
        n_markets=("id", "size"),
        n_unique_tickers=("ticker", "nunique"),
        stock_ticker=("ticker", "first")
    )
    .reset_index()
)

ticker_summary = {
    "n_events_in_market_table": len(ticker_check),
    "events_with_single_market_ticker": (ticker_check["n_unique_tickers"] == 1).sum(),
    "events_with_inconsistent_market_ticker": (ticker_check["n_unique_tickers"] != 1).sum(),
}

ticker_summary

{'n_events_in_market_table': 624,
 'events_with_single_market_ticker': 624,
 'events_with_inconsistent_market_ticker': 0}

### 4.3 Add authoritative stock ticker to the event-level table

If the consistency checks pass, we add a new column `stock_ticker` to `events_data`, derived from the market table.

This keeps the original event-level columns unchanged while making the Bloomberg-relevant stock ticker explicit.

In [23]:
# Add authoritative stock ticker from markets_data to events_data
event_stock_ticker = ticker_check[["event_id", "stock_ticker"]].copy()

events_data = (
    events_data
    .merge(
        event_stock_ticker,
        left_on="id",
        right_on="event_id",
        how="left"
    )
    .drop(columns=["event_id"])
)

events_data.loc[:4, ["id", "ticker", "stock_ticker", "slug", "title"]]

,id,ticker,stock_ticker,slug,title
0,72220,aapl-above-in-november-2025,AAPL,aapl-above-in-november-2025,Will Apple (AAPL) close above ___ end of Novem...
1,72221,msft-above-in-november-2025,MSFT,msft-above-in-november-2025,Will Microsoft (MSFT) close above ___ end of N...
2,72222,amzn-above-in-november-2025,AMZN,amzn-above-in-november-2025,Will Amazon (AMZN) close above ___ end of Nove...
3,72223,googl-above-in-november-2025,GOOGL,googl-above-in-november-2025,Will Google (GOOGL) close above ___ end of Nov...
4,72224,meta-above-in-november-2025,META,meta-above-in-november-2025,Will Meta (META) close above ___ end of November?


## Interpretation

The checks above show:

- all markets under an event share the same end timestamp,
- the event end timestamp matches the market end timestamp,
- and all markets under an event share the same stock ticker,

therefore we can safely use:

- `events_data` for event timing,
- `markets_data` for payoff bounds,
- and `events_data["stock_ticker"]` (derived from `markets_data`) as the authoritative stock ticker for Bloomberg planning.

## Filter to usable events for Bloomberg daily option history

Bloomberg daily option history is only available starting around 2025-12-18.  
To keep the sample usable for the later options study, we apply two timing filters:

1. **Resolved by the analysis date**  
   We confirm that events are already resolved as of 2026-04-17.

2. **Sufficiently late end date**  
   We keep only events whose ET end date is on or after 2025-12-20.

After filtering the event table, we keep only markets whose `event_id` belongs to the remaining events.

This section produces:

- a filtered event table
- a filtered market table
- summary counts before and after filtering
- a ticker-level summary of the filtered event sample

In [24]:
# --------------------------------------------------
# Part 5.1: define date cutoffs
# --------------------------------------------------
cutoff_end_date_et = pd.Timestamp("2025-12-20").date()
analysis_asof_date_et = pd.Timestamp("2026-04-17").date()

print("cutoff_end_date_et   :", cutoff_end_date_et)
print("analysis_asof_date_et:", analysis_asof_date_et)

cutoff_end_date_et   : 2025-12-20
analysis_asof_date_et: 2026-04-17


### 5.1 Confirm that all events are already resolved

We first verify that all events in the current event table have ET end dates earlier than 2026-04-17.

In [25]:
events_resolution_check = events_data.copy()

events_resolution_check["is_resolved_by_2026_04_17"] = (
    events_resolution_check["endDate_et"].dt.date < analysis_asof_date_et
)

n_events_total = len(events_resolution_check)
n_resolved = int(events_resolution_check["is_resolved_by_2026_04_17"].sum())
n_not_resolved = n_events_total - n_resolved

print("Total events           :", n_events_total)
print("Resolved by 2026-04-17:", n_resolved)
print("Not resolved           :", n_not_resolved)

Total events           : 624
Resolved by 2026-04-17: 624
Not resolved           : 0


### 5.2 Filter by usable event end date

We now keep only events whose ET end date is on or after 2025-12-20.

This is a practical sample restriction motivated by Bloomberg daily option-history availability.

In [26]:
events_filtered = events_data.loc[
    events_data["endDate_et"].dt.date >= cutoff_end_date_et
].copy()

keep_event_ids = set(events_filtered["id"].astype("string"))

markets_filtered = markets_data.loc[
    markets_data["event_id"].astype("string").isin(keep_event_ids)
].copy()

print("Before filter:")
print("  events :", len(events_data))
print("  markets:", len(markets_data))

print("\nAfter filter (endDate_et >= 2025-12-20):")
print("  events :", len(events_filtered))
print("  markets:", len(markets_filtered))

Before filter:
  events : 624
  markets: 5907

After filter (endDate_et >= 2025-12-20):
  events : 497
  markets: 4409


### 5.3 Summary of filtered sample by stock ticker

We summarize the filtered event sample using the authoritative stock ticker derived from the market table.

In [27]:
ticker_summary_filtered = (
    events_filtered.groupby("stock_ticker")
    .size()
    .reset_index(name="n_events")
    .sort_values(["n_events", "stock_ticker"], ascending=[False, True])
    .reset_index(drop=True)
)

ticker_summary_filtered

,stock_ticker,n_events
0,NVDA,64
1,AAPL,61
2,AMZN,61
3,MSFT,58
4,GOOGL,56
5,TSLA,56
6,META,54
7,NFLX,30
8,OPEN,29
9,PLTR,28


## Save filtered datasets

We save the filtered event and market tables so later sections can work directly with the usable Bloomberg sample.

In [28]:
events_filtered_to_save = prepare_events_for_save(events_filtered)
markets_filtered_to_save = prepare_markets_for_save(markets_filtered)

events_filtered_to_save.to_csv("stocks_events_close_only_filtered.csv", index=False)
markets_filtered_to_save.to_csv("stocks_markets_close_only_filtered.csv", index=False)
ticker_summary_filtered.to_csv("filtered_event_ticker_summary.csv", index=False)

print("Saved:")
print("- stocks_events_close_only_filtered.csv")
print("- stocks_markets_close_only_filtered.csv")
print("- filtered_event_ticker_summary.csv")

Saved:
- stocks_events_close_only_filtered.csv
- stocks_markets_close_only_filtered.csv
- filtered_event_ticker_summary.csv


In [29]:
events_data = events_filtered_to_save.copy()
markets_data = markets_filtered_to_save.copy()

print("\nCurrent working shapes:")
print("  events_data :", events_data.shape)
print("  markets_data:", markets_data.shape)


Current working shapes:
  events_data : (497, 20)
  markets_data: (4409, 18)


In [30]:
events_data

,id,ticker,slug,title,description,category,startDate,creationDate,endDate,resolutionSource,closed,archived,volume,volume_num,resolved_flag,Is_Hit_Market,startDate_et,creationDate_et,endDate_et,stock_ticker
47,86643,what-will-amzn-close-at-in-2025,what-will-amzn-close-at-in-2025,What will Amazon (AMZN) close at in 2025?,This market will resolve according to the offi...,Stocks,2025-11-24 22:12:30.742221+00:00,2025-11-24 22:12:30.742214+00:00,2025-12-31 23:59:59+00:00,NaN,True,False,130072.538872,130072.538872,True,False,2025-11-24 17:12:30.742221-05:00,2025-11-24 17:12:30.742214-05:00,2025-12-31 18:59:59-05:00,AMZN
48,86644,what-will-nflx-close-at-in-2025,what-will-nflx-close-at-in-2025,What will Netflix (NFLX) close at in 2025?,This market will resolve according to the offi...,Stocks,2025-11-24 22:00:42.034201+00:00,2025-11-24 22:00:42.034195+00:00,2025-12-31 23:59:59+00:00,NaN,True,False,252989.498871,252989.498871,True,False,2025-11-24 17:00:42.034201-05:00,2025-11-24 17:00:42.034195-05:00,2025-12-31 18:59:59-05:00,NFLX
73,89531,what-will-tesla-tsla-close-at-in-2025,what-will-tesla-tsla-close-at-in-2025,What will Tesla (TSLA) close at in 2025?,This market will resolve according to the offi...,Stocks,2025-11-24 22:12:30.736651+00:00,2025-11-24 22:12:30.736642+00:00,2025-12-31 00:00:00+00:00,NaN,True,False,264942.177680,264942.177680,True,False,2025-11-24 17:12:30.736651-05:00,2025-11-24 17:12:30.736642-05:00,2025-12-30 19:00:00-05:00,TSLA
74,89532,what-will-google-googl-close-at-in-2025,what-will-google-googl-close-at-in-2025,What will Google (GOOGL) close at in 2025?,This market will resolve according to the offi...,Stocks,2025-11-24 22:12:30.739461+00:00,2025-11-24 22:12:30.739454+00:00,2025-12-31 00:00:00+00:00,NaN,True,False,288227.869741,288227.869741,True,False,2025-11-24 17:12:30.739461-05:00,2025-11-24 17:12:30.739454-05:00,2025-12-30 19:00:00-05:00,GOOGL
75,89541,what-will-nvidia-nvda-close-at-in-2025,what-will-nvidia-nvda-close-at-in-2025,What will NVIDIA (NVDA) close at in 2025?,This market will resolve according to the offi...,Stocks,2025-11-24 22:12:30.755480+00:00,2025-11-24 22:12:30.755473+00:00,2025-12-31 00:00:00+00:00,NaN,True,False,385224.088320,385224.088320,True,False,2025-11-24 17:12:30.755480-05:00,2025-11-24 17:12:30.755473-05:00,2025-12-30 19:00:00-05:00,NVDA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619,328743,nvda-close-above-on-april-1-2026,nvda-close-above-on-april-1-2026,NVIDIA (NVDA) closes above ___ on April 1?,"This market will resolve to ""Yes"" if the offic...",Stocks,2026-03-31 12:26:34.067240+00:00,2026-04-01 13:30:00+00:00,2026-04-01 20:00:00+00:00,https://finance.yahoo.com/quote/NVDA/history,True,False,14942.749623,14942.749623,True,False,2026-03-31 08:26:34.067240-04:00,2026-04-01 09:30:00-04:00,2026-04-01 16:00:00-04:00,NVDA
620,332157,amzn-close-above-on-april-2-2026,amzn-close-above-on-april-2-2026,Amazon (AMZN) closes above ___ on April 2?,"This market will resolve to ""Yes"" if the offic...",Stocks,2026-04-01 12:05:48.555541+00:00,2026-04-02 13:30:00+00:00,2026-04-02 20:00:00+00:00,https://finance.yahoo.com/quote/AMZN/history,True,False,19846.563208,19846.563208,True,False,2026-04-01 08:05:48.555541-04:00,2026-04-02 09:30:00-04:00,2026-04-02 16:00:00-04:00,AMZN
621,332172,nvda-close-above-on-april-2-2026,nvda-close-above-on-april-2-2026,NVIDIA (NVDA) closes above ___ on April 2?,"This market will resolve to ""Yes"" if the offic...",Stocks,2026-04-01 12:05:50.242464+00:00,2026-04-02 13:30:00+00:00,2026-04-02 20:00:00+00:00,https://finance.yahoo.com/quote/NVDA/history,True,False,11698.192693,11698.192693,True,False,2026-04-01 08:05:50.242464-04:00,2026-04-02 09:30:00-04:00,2026-04-02 16:00:00-04:00,NVDA
622,335714,aapl-close-above-on-april-6-2026,aapl-close-above-on-april-6-2026,Apple (AAPL) closes above ___ on April 6?,"This market will resolve to ""Yes"" if the offic...",Stocks,2026-04-02 12:06:44.246048+00:00,2026-04-06 13:30:00+00:00,2026-

## Pilot Bloomberg event: `aapl-above-in-february-2026`

We use the event with slug:

`aapl-above-in-february-2026`

as a pilot case for Bloomberg extraction.

The goal is to build a compact event-level summary that tells us:

- the authoritative stock ticker
- the event start and end timestamps
- the first trading day after event start
- the target close trading day
- the number of related Polymarket markets
- the relevant payoff boundaries / strikes

We then use this event to test the Bloomberg workflow manually before scaling to the full sample.

In [31]:
selected_slug = "aapl-above-in-february-2026"

pilot_event = events_data.loc[events_data["slug"] == selected_slug].copy()
pilot_markets = markets_data.loc[markets_data["event_slug"] == selected_slug].copy()

print("pilot_event shape :", pilot_event.shape)
print("pilot_markets shape:", pilot_markets.shape)

pilot_event.head(3), pilot_markets.head(3)

pilot_event shape : (1, 20)
pilot_markets shape: (13, 18)


(         id                       ticker                         slug  \
 323  193924  aapl-above-in-february-2026  aapl-above-in-february-2026   
 
                                                  title  \
 323  Will Apple (AAPL) close above ___ end of Febru...   
 
                                            description category  \
 323  This market will resolve to "Yes" if the offic...   Stocks   
 
                            startDate              creationDate  \
 323 2026-01-30 23:19:45.471926+00:00 2026-02-01 05:00:00+00:00   
 
                       endDate                              resolutionSource  \
 323 2026-02-27 21:00:00+00:00  https://finance.yahoo.com/quote/AAPL/history   
 
      closed  archived         volume     volume_num  resolved_flag  \
 323    True     False  192980.816273  192980.816273           True   
 
      Is_Hit_Market                     startDate_et           creationDate_et  \
 323          False 2026-01-30 18:19:45.471926-05:00 2026-02-01 00:0

In [32]:
assert len(pilot_event) == 1, f"Expected 1 event row, got {len(pilot_event)}"
assert pilot_markets["event_id"].nunique() == 1, "pilot_markets contains more than one event_id"
assert pilot_markets["ticker"].nunique() == 1, "pilot_markets has inconsistent stock tickers"

pilot_event = pilot_event.iloc[[0]].copy()
pilot_markets = pilot_markets.sort_values(["lower_bound", "upper_bound", "id"]).reset_index(drop=True)

print("Selected event_id   :", pilot_event["id"].iloc[0])
print("Selected stock_ticker:", pilot_event["stock_ticker"].iloc[0])
print("Number of markets   :", len(pilot_markets))

Selected event_id   : 193924
Selected stock_ticker: AAPL
Number of markets   : 13


## Build a compact event summary for the Bloomberg pilot

We summarize the event timing and relevant payoff boundaries.

For this specific event, the relevant boundaries are the finite values appearing in:

- `lower_bound`
- `upper_bound`

Because this is an “above in February” event, we expect most or all markets to be of the form `[x, inf)`, so the main relevant boundaries will likely be the lower bounds.

In [33]:
def next_business_day_strict(ts):
    d = pd.Timestamp(ts.date())
    return (d + pd.offsets.BDay(1)).date()

def target_close_trading_day(ts):
    return pd.Timestamp(ts.date()).date()

def business_days_inclusive(start_date, end_date):
    if pd.isna(start_date) or pd.isna(end_date):
        return np.nan
    if start_date > end_date:
        return 0
    return len(pd.bdate_range(start=start_date, end=end_date))

def unique_sorted_finite(x):
    vals = pd.to_numeric(pd.Series(x), errors="coerce")
    vals = vals[np.isfinite(vals)]
    return sorted(vals.unique().tolist())

pilot_start_ts_et = pilot_event["startDate_et"].iloc[0]
pilot_end_ts_et = pilot_event["endDate_et"].iloc[0]

pilot_summary = pd.DataFrame([{
    "event_id": pilot_event["id"].iloc[0],
    "slug": pilot_event["slug"].iloc[0],
    "title": pilot_event["title"].iloc[0],
    "stock_ticker": pilot_event["stock_ticker"].iloc[0],
    "start_ts_et": pilot_start_ts_et,
    "start_date_et": pilot_start_ts_et.date(),
    "first_trading_day_after_start_et": next_business_day_strict(pilot_start_ts_et),
    "end_ts_et": pilot_end_ts_et,
    "end_date_et": pilot_end_ts_et.date(),
    "target_close_trading_day_et": target_close_trading_day(pilot_end_ts_et),
    "calendar_days_start_to_end": (pilot_end_ts_et.normalize() - pilot_start_ts_et.normalize()).days,
    "trading_days_first_after_start_to_target": business_days_inclusive(
        next_business_day_strict(pilot_start_ts_et),
        target_close_trading_day(pilot_end_ts_et)
    ),
    "n_markets": len(pilot_markets),
    "n_x_inf": (pilot_markets["interval_type"] == "[x, inf)").sum(),
    "n_0_x": (pilot_markets["interval_type"] == "[0, x]").sum(),
    "n_x_y": (pilot_markets["interval_type"] == "[x, y]").sum(),
    "n_other": (pilot_markets["interval_type"] == "other").sum(),
    "lower_bounds": [unique_sorted_finite(pilot_markets["lower_bound"])],
    "upper_bounds": [unique_sorted_finite(pilot_markets["upper_bound"])],
    "pooled_finite_strikes": [unique_sorted_finite(
        pd.concat([
            pilot_markets["lower_bound"],
            pilot_markets["upper_bound"]
        ], ignore_index=True)
    )],
    "resolutionSource": pilot_event["resolutionSource"].iloc[0],
}])

pilot_summary

,event_id,slug,title,stock_ticker,start_ts_et,start_date_et,first_trading_day_after_start_et,end_ts_et,end_date_et,target_close_trading_day_et,...,trading_days_first_after_start_to_target,n_markets,n_x_inf,n_0_x,n_x_y,n_other,lower_bounds,upper_bounds,pooled_finite_strikes,resolutionSource
0,193924,aapl-above-in-february-2026,Will Apple (AAPL) close above ___ end of Febru...,AAPL,2026-01-30 18:19:45.471926-05:00,2026-01-30,2026-02-02,2026-02-27 16:00:00-05:00,2026-02-27,2026-02-27,...,20,13,13,0,0,0,"[[200.0, 210.0, 220.0, 230.0, 240.0, 250.0, 26...",[[]],"[[200.0, 210.0, 220.0, 230.0, 240.0, 250.0, 26...",https://finance.yahoo.com/quote/AAPL/history


In [34]:
pilot_markets[[
    "id", "question", "interval_type", "lower_bound", "upper_bound"
]]

,id,question,interval_type,lower_bound,upper_bound
0,1301557,Will Apple (AAPL) close above $200 end of Febr...,"[x, inf)",200.0,inf
1,1301560,Will Apple (AAPL) close above $210 end of Febr...,"[x, inf)",210.0,inf
2,1301563,Will Apple (AAPL) close above $220 end of Febr...,"[x, inf)",220.0,inf
3,1301566,Will Apple (AAPL) close above $230 end of Febr...,"[x, inf)",230.0,inf
4,1301569,Will Apple (AAPL) close above $240 end of Febr...,"[x, inf)",240.0,inf
5,1301572,Will Apple (AAPL) close above $250 end of Febr...,"[x, inf)",250.0,inf
6,1301575,Will Apple (AAPL) close above $260 end of Febr...,"[x, inf)",260.0,inf
7,1301580,Will Apple (AAPL) close above $270 end of Febr...,"[x, inf)",270.0,inf
8,1301583,Will Apple (AAPL) close above $280 end of Febr...,"[x, inf)",280.0,inf
9,1301587,Will Apple (AAPL) close above $290 end of Febr...,"[x, inf)",290.0,inf


## Build a strike list for Bloomberg checking

For this event, the first Bloomberg task is to determine which option expiries are available near the Polymarket target date.

The second task is to determine which listed strikes are available for the chosen expiry.

This table lists the finite Polymarket payoff boundaries that are relevant for option selection.

In [36]:
pilot_strikes = pd.DataFrame({
    "stock_ticker": pilot_event["stock_ticker"].iloc[0],
    "event_id": pilot_event["id"].iloc[0],
    "slug": pilot_event["slug"].iloc[0],
    "pm_target_date_et": target_close_trading_day(pilot_end_ts_et),
    "strike": unique_sorted_finite(
        pd.concat([
            pilot_markets["lower_bound"],
            pilot_markets["upper_bound"]
        ], ignore_index=True)
    )
})

pilot_strikes

,stock_ticker,event_id,slug,pm_target_date_et,strike
0,AAPL,193924,aapl-above-in-february-2026,2026-02-27,200.0
1,AAPL,193924,aapl-above-in-february-2026,2026-02-27,210.0
2,AAPL,193924,aapl-above-in-february-2026,2026-02-27,220.0
3,AAPL,193924,aapl-above-in-february-2026,2026-02-27,230.0
4,AAPL,193924,aapl-above-in-february-2026,2026-02-27,240.0
5,AAPL,193924,aapl-above-in-february-2026,2026-02-27,250.0
6,AAPL,193924,aapl-above-in-february-2026,2026-02-27,260.0
7,AAPL,193924,aapl-above-in-february-2026,2026-02-27,270.0
8,AAPL,193924,aapl-above-in-february-2026,2026-02-27,280.0
9,AAPL,193924,aapl-above-in-february-2026,2026-02-27,290.0


In [37]:
pilot_strikes.to_csv("pilot_aapl_above_in_february_2026_strikes.csv", index=False)
pilot_summary.to_csv("pilot_aapl_above_in_february_2026_summary.csv", index=False)

print("Saved:")
print("- pilot_aapl_above_in_february_2026_summary.csv")
print("- pilot_aapl_above_in_february_2026_strikes.csv")

Saved:
- pilot_aapl_above_in_february_2026_summary.csv
- pilot_aapl_above_in_february_2026_strikes.csv


## What to test on Bloomberg for this pilot

For this event, we want to answer the following operational questions:

1. What option expiration dates for AAPL are available near the Polymarket target date?
2. Can Bloomberg Excel or the terminal identify these expiries and strikes in a way that is usable for historical extraction?
3. Once an expiry is selected, can we extract daily option data for the relevant strikes over the event window?

For now, we keep the maturity rule simple:

- keep the event if there is an exact or near-by option expiry,
- otherwise reconsider later.

At this stage, the main purpose is to test feasibility, not yet to optimize the full-scale extraction workflow.

In [93]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Read common expiry dates
# --------------------------------------------------
exp_df = pd.read_excel("option expire dates.xlsx", header=None)

# first column contains the dates
common_expiries = pd.to_datetime(exp_df.iloc[:, 0], errors="coerce").dt.date.tolist()
common_expiries = [d for d in common_expiries if pd.notna(d)]
common_expiries = sorted(pd.unique(common_expiries))

print("Number of common expiries:", len(common_expiries))
print(common_expiries[:10])

Number of common expiries: 35
[datetime.date(2025, 12, 26), datetime.date(2026, 1, 2), datetime.date(2026, 1, 9), datetime.date(2026, 1, 16), datetime.date(2026, 1, 23), datetime.date(2026, 1, 30), datetime.date(2026, 2, 2), datetime.date(2026, 2, 4), datetime.date(2026, 2, 6), datetime.date(2026, 2, 9)]


/var/folders/6d/4cbpc7vx1f5d6762pmx36swm0000gn/T/ipykernel_90099/2222121398.py:12: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_expiries = sorted(pd.unique(common_expiries))


In [95]:
# --------------------------------------------------
# 2. Read per-stock test strikes
#    Each column = stock ticker
#    Each non-missing value = a strike to test
# --------------------------------------------------
strike_df = pd.read_excel("stock tickers.xlsx")

print(strike_df.shape)
display(strike_df)

(3, 10)


,AAPL,AMZN,GOOGL,META,MSFT,NFLX,NVDA,OPEN,PLTR,TSLA
0,270.0,250.0,340.0,650.0,400.0,98.0,180.0,6,146,400.0
1,NaN,NaN,300.0,500.0,NaN,90.0,NaN,5,160,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,168,NaN


In [96]:
# --------------------------------------------------
# 3. Convert the strike sheet to long format
# --------------------------------------------------
stock_strikes_long = (
    strike_df
    .melt(var_name="stock_ticker", value_name="strike")
    .dropna(subset=["strike"])
    .copy()
)

stock_strikes_long["strike"] = pd.to_numeric(stock_strikes_long["strike"], errors="coerce")
stock_strikes_long = stock_strikes_long.dropna(subset=["strike"]).copy()

# remove duplicates and sort
stock_strikes_long = (
    stock_strikes_long
    .drop_duplicates()
    .sort_values(["stock_ticker", "strike"])
    .reset_index(drop=True)
)

display(stock_strikes_long)

,stock_ticker,strike
0,AAPL,270.0
1,AMZN,250.0
2,GOOGL,300.0
3,GOOGL,340.0
4,META,500.0
5,META,650.0
6,MSFT,400.0
7,NFLX,90.0
8,NFLX,98.0
9,NVDA,180.0


In [98]:
# --------------------------------------------------
# 4. Generate stock × expiry × strike
# --------------------------------------------------
rows = []

for _, r in stock_strikes_long.iterrows():
    stock = r["stock_ticker"]
    strike = r["strike"]

    for expiry in common_expiries:
        rows.append({
            "stock_ticker": stock,
            "option_expiry": expiry,
            "strike": strike,
        })

option_universe_test_strikes = pd.DataFrame(rows)

# calls only for now
option_universe_test_strikes["cp"] = "C"

option_universe_test_strikes["bbg_ticker"] = (
    option_universe_test_strikes["stock_ticker"]
    + " US "
    + pd.to_datetime(option_universe_test_strikes["option_expiry"]).dt.strftime("%m/%d/%y")
    + " "
    + option_universe_test_strikes["cp"]
    + option_universe_test_strikes["strike"].map(lambda x: f"{x:g}")
    + " Equity"
)

option_universe_test_strikes = option_universe_test_strikes.sort_values(
    ["stock_ticker", "option_expiry", "strike"]
).reset_index(drop=True)

print(option_universe_test_strikes.shape)
display(option_universe_test_strikes.head(30))

(595, 5)


,stock_ticker,option_expiry,strike,cp,bbg_ticker
0,AAPL,2025-12-26,270.0,C,AAPL US 12/26/25 C270 Equity
1,AAPL,2026-01-02,270.0,C,AAPL US 01/02/26 C270 Equity
2,AAPL,2026-01-09,270.0,C,AAPL US 01/09/26 C270 Equity
3,AAPL,2026-01-16,270.0,C,AAPL US 01/16/26 C270 Equity
4,AAPL,2026-01-23,270.0,C,AAPL US 01/23/26 C270 Equity
5,AAPL,2026-01-30,270.0,C,AAPL US 01/30/26 C270 Equity
6,AAPL,2026-02-02,270.0,C,AAPL US 02/02/26 C270 Equity
7,AAPL,2026-02-04,270.0,C,AAPL US 02/04/26 C270 Equity
8,AAPL,2026-02-06,270.0,C,AAPL US 02/06/26 C270 Equity
9,AAPL,2026-02-09,270.0,C,AAPL US 02/09/26 C270 Equity


In [99]:
# --------------------------------------------------
# 5. Save
# --------------------------------------------------
option_universe_test_strikes.to_csv("option_universe_test_strikes_all_stocks.csv", index=False)
#option_universe_test_strikes.to_excel("option_universe_test_strikes_all_stocks.xlsx", index=False)

print("Saved:")
print("- option_universe_test_strikes_all_stocks.csv")
#print("- option_universe_test_strikes_all_stocks.xlsx")

Saved:
- option_universe_test_strikes_all_stocks.csv


step 2

In [118]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Read Step 1 result
# -----------------------------
# Use one of these:
expire_exist = pd.read_csv("option_universe_test_strikes_all_stocks (1).csv")
# expire_exist = pd.read_excel("option_universe_test_strikes_all_stocks.xlsx")

print(expire_exist.shape)
print(expire_exist.columns.tolist())
expire_exist.head()

(595, 7)
['stock_ticker', 'option_expiry', 'strike', 'cp', 'bbg_ticker', 'Option Expire Date', 'Expire Date Exist']


,stock_ticker,option_expiry,strike,cp,bbg_ticker,Option Expire Date,Expire Date Exist
0,AAPL,12/26/2025,270,C,AAPL US 12/26/25 C270 Equity,12/26/2025,True
1,AAPL,1/2/2026,270,C,AAPL US 01/02/26 C270 Equity,1/2/2026,True
2,AAPL,1/9/2026,270,C,AAPL US 01/09/26 C270 Equity,1/9/2026,True
3,AAPL,1/16/2026,270,C,AAPL US 01/16/26 C270 Equity,1/16/2026,True
4,AAPL,1/23/2026,270,C,AAPL US 01/23/26 C270 Equity,1/23/2026,True


In [119]:
# -----------------------------
# 2. Clean columns
# -----------------------------
expire_exist = expire_exist.copy()

# Parse option expiry
expire_exist["option_expiry"] = pd.to_datetime(
    expire_exist["option_expiry"], errors="coerce"
).dt.date

# If Excel also returned a date column from Bloomberg, parse it too
if "Option Expire" in expire_exist.columns:
    expire_exist["Option Expire"] = pd.to_datetime(
        expire_exist["Option Expire"], errors="coerce"
    ).dt.date

# Clean existence flag
expire_exist["Expire Date Exist"] = (
    expire_exist["Expire Date Exist"]
    .astype(str)
    .str.strip()
    .str.upper()
)

expire_exist["expire_exists"] = expire_exist["Expire Date Exist"].isin(["TRUE", "YES", "Y", "1"])

# Keep only stock-expiry pairs that exist
available_expiries = (
    expire_exist.loc[expire_exist["expire_exists"], ["stock_ticker", "option_expiry"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["stock_ticker", "option_expiry"])
    .reset_index(drop=True)
)

print("Available stock-expiry pairs:", len(available_expiries))
available_expiries.head(20)

Available stock-expiry pairs: 292


,stock_ticker,option_expiry
0,AAPL,2025-12-26
1,AAPL,2026-01-02
2,AAPL,2026-01-09
3,AAPL,2026-01-16
4,AAPL,2026-01-23
5,AAPL,2026-01-30
6,AAPL,2026-02-02
7,AAPL,2026-02-04
8,AAPL,2026-02-06
9,AAPL,2026-02-09


In [120]:
available_expiry_list_by_stock = (
    available_expiries.groupby("stock_ticker")["option_expiry"]
    .apply(list)
    .reset_index(name="available_expiry_list")
)

available_expiry_list_by_stock

,stock_ticker,available_expiry_list
0,AAPL,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."
1,AMZN,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."
2,GOOGL,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."
3,META,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."
4,MSFT,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."
5,NFLX,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."
6,NVDA,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."
7,OPEN,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."
8,PLTR,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."
9,TSLA,"[2025-12-26, 2026-01-02, 2026-01-09, 2026-01-1..."


In [121]:
# -----------------------------
# 3. Prepare event-level table
# -----------------------------
event_expiry_match = events_data[[
    "id", "slug", "title", "stock_ticker", "endDate_et"
]].copy()

event_expiry_match = event_expiry_match.rename(columns={"id": "event_id"})
event_expiry_match["pm_enddate"] = pd.to_datetime(event_expiry_match["endDate_et"]).dt.date

event_expiry_match.head()

,event_id,slug,title,stock_ticker,endDate_et,pm_enddate
47,86643,what-will-amzn-close-at-in-2025,What will Amazon (AMZN) close at in 2025?,AMZN,2025-12-31 18:59:59-05:00,2025-12-31
48,86644,what-will-nflx-close-at-in-2025,What will Netflix (NFLX) close at in 2025?,NFLX,2025-12-31 18:59:59-05:00,2025-12-31
73,89531,what-will-tesla-tsla-close-at-in-2025,What will Tesla (TSLA) close at in 2025?,TSLA,2025-12-30 19:00:00-05:00,2025-12-30
74,89532,what-will-google-googl-close-at-in-2025,What will Google (GOOGL) close at in 2025?,GOOGL,2025-12-30 19:00:00-05:00,2025-12-30
75,89541,what-will-nvidia-nvda-close-at-in-2025,What will NVIDIA (NVDA) close at in 2025?,NVDA,2025-12-30 19:00:00-05:00,2025-12-30


In [122]:
# -----------------------------
# 4. Match nearest expiry
# -----------------------------
def find_closest_expiry(stock, pm_enddate, available_df):
    stock_exp = available_df.loc[
        available_df["stock_ticker"] == stock, "option_expiry"
    ].dropna().tolist()

    if len(stock_exp) == 0:
        return pd.Series({
            "matched_option_expiry": pd.NaT,
            "mismatch_days": np.nan,
            "abs_mismatch_days": np.nan
        })

    pm_ts = pd.Timestamp(pm_enddate)
    exp_ts = pd.to_datetime(stock_exp)

    diff_days = (exp_ts - pm_ts).days
    abs_diff_days = np.abs(diff_days)

    # tie-break rule:
    # 1. minimize absolute mismatch
    # 2. if tied, prefer earlier expiry (more negative mismatch)
    best_idx = np.lexsort((diff_days, abs_diff_days))[0]

    best_expiry = exp_ts[best_idx].date()
    best_mismatch = int(diff_days[best_idx])

    return pd.Series({
        "matched_option_expiry": best_expiry,
        "mismatch_days": best_mismatch,
        "abs_mismatch_days": abs(best_mismatch)
    })

event_expiry_match[[
    "matched_option_expiry",
    "mismatch_days",
    "abs_mismatch_days"
]] = event_expiry_match.apply(
    lambda row: find_closest_expiry(
        row["stock_ticker"],
        row["pm_enddate"],
        available_expiries
    ),
    axis=1
)

event_expiry_match.head(20)

,event_id,slug,title,stock_ticker,endDate_et,pm_enddate,matched_option_expiry,mismatch_days,abs_mismatch_days
47,86643,what-will-amzn-close-at-in-2025,What will Amazon (AMZN) close at in 2025?,AMZN,2025-12-31 18:59:59-05:00,2025-12-31,2026-01-02,2,2
48,86644,what-will-nflx-close-at-in-2025,What will Netflix (NFLX) close at in 2025?,NFLX,2025-12-31 18:59:59-05:00,2025-12-31,2026-01-02,2,2
73,89531,what-will-tesla-tsla-close-at-in-2025,What will Tesla (TSLA) close at in 2025?,TSLA,2025-12-30 19:00:00-05:00,2025-12-30,2026-01-02,3,3
74,89532,what-will-google-googl-close-at-in-2025,What will Google (GOOGL) close at in 2025?,GOOGL,2025-12-30 19:00:00-05:00,2025-12-30,2026-01-02,3,3
75,89541,what-will-nvidia-nvda-close-at-in-2025,What will NVIDIA (NVDA) close at in 2025?,NVDA,2025-12-30 19:00:00-05:00,2025-12-30,2026-01-02,3,3
76,89542,what-will-microsoft-msft-close-at-in-2025,What will Microsoft (MSFT) close at in 2025?,MSFT,2025-12-30 19:00:00-05:00,2025-12-30,2026-01-02,3,3
77,89543,what-will-apple-aapl-close-at-in-2025,What will Apple (AAPL) close at in 2025?,AAPL,2025-12-30 19:00:00-05:00,2025-12-30,2026-01-02,3,3
79,92859,aapl-above-in-december-2025,Will Apple (AAPL) close above ___ end of Decem...,AAPL,2025-12-31 16:00:00-05:00,2025-12-31,2026-01-02,2,2
81,92863,msft-above-in-december-2025,Will Microsoft (MSFT) close above ___ end of D...,MSFT,2025-12-31 16:00:00-05:00,2025-12-31,2026-01-02,2,2
85,92868,amzn-above-in-december-2025,Will Amazon (AMZN) close above ___ end of Dece...,AMZN,2025-12-31 16:00:00-05:00,2025-12-31,2026-01-02,2,2


In [123]:
# -----------------------------
# 5. Summaries
# -----------------------------
print("Events with no matched expiry:",
      event_expiry_match["matched_option_expiry"].isna().sum())

mismatch_summary = (
    event_expiry_match["abs_mismatch_days"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("abs_mismatch_days")
    .reset_index(name="n_events")
)

mismatch_summary

Events with no matched expiry: 0


,abs_mismatch_days,n_events
0,0,355
1,1,100
2,2,25
3,3,17


In [106]:
for k in [0, 1, 2, 3, 5]:
    n_keep = (event_expiry_match["abs_mismatch_days"] <= k).sum()
    print(f"abs_mismatch_days <= {k}: {n_keep} events")

abs_mismatch_days <= 0: 355 events
abs_mismatch_days <= 1: 455 events
abs_mismatch_days <= 2: 480 events
abs_mismatch_days <= 3: 497 events
abs_mismatch_days <= 5: 497 events


## Step 3: Build stock × option expiry × strike universe

We now move from event-level expiry matching to the option universe we may need to test.

For each event, we already know the closest available option expiry.  
In this step:

1. Keep the broader extraction universe with `abs_mismatch_days <= 1`
2. For each event, collect the relevant PM strike range
3. Aggregate to unique `stock_ticker × matched_option_expiry`
4. Expand into one row per `stock × option_expiry × strike`

This table will later be exported to Excel, where we construct Bloomberg option tickers and test whether the options actually exist.

In [124]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# Step 3.1: stock-specific strike steps
# --------------------------------------------------
stock_strike_steps = {
    "AAPL": 2.5,
    "AMZN": 2.5,
    "GOOGL": 2.5,
    "META": 2.5,
    "MSFT": 2.5,
    "NFLX": 1.0,
    "NVDA": 2.5,
    "OPEN": 0.5,
    "PLTR": 0.5,
    "TSLA": 2.5,
}

stock_strike_steps_df = pd.DataFrame(
    list(stock_strike_steps.items()),
    columns=["stock_ticker", "strike_step"]
)

In [125]:
# --------------------------------------------------
# Step 3.2: event-level PM strike range
# --------------------------------------------------
event_strikes_long = pd.concat([
    markets_data[["event_id", "lower_bound"]].rename(columns={"lower_bound": "strike"}),
    markets_data[["event_id", "upper_bound"]].rename(columns={"upper_bound": "strike"}),
], ignore_index=True)

event_strikes_long["strike"] = pd.to_numeric(event_strikes_long["strike"], errors="coerce")
event_strikes_long = event_strikes_long[
    np.isfinite(event_strikes_long["strike"]) & (event_strikes_long["strike"] > 0)
].copy()

event_strike_range = (
    event_strikes_long.groupby("event_id")["strike"]
    .agg(
        min_pm_strike="min",
        max_pm_strike="max"
    )
    .reset_index()
)

event_strike_range.head()

,event_id,min_pm_strike,max_pm_strike
0,86643,200.0,250.0
1,86644,85.0,125.0
2,89531,300.0,500.0
3,89532,250.0,400.0
4,89541,160.0,220.0


In [126]:
# --------------------------------------------------
# Step 3.3: use the Step 2 output directly
# IMPORTANT: do NOT recreate event_expiry_match from events_data
# --------------------------------------------------
# event_expiry_match should already contain:
# matched_option_expiry, mismatch_days, abs_mismatch_days

required_cols = [
    "event_id", "stock_ticker", "pm_enddate",
    "matched_option_expiry", "mismatch_days", "abs_mismatch_days"
]

missing_cols = [c for c in required_cols if c not in event_expiry_match.columns]
if missing_cols:
    raise ValueError(f"event_expiry_match is missing columns: {missing_cols}")

event_expiry_with_strikes = (
    event_expiry_match
    .merge(event_strike_range, on="event_id", how="left")
    .copy()
)

event_expiry_with_strikes.head()

,event_id,slug,title,stock_ticker,endDate_et,pm_enddate,matched_option_expiry,mismatch_days,abs_mismatch_days,min_pm_strike,max_pm_strike
0,86643,what-will-amzn-close-at-in-2025,What will Amazon (AMZN) close at in 2025?,AMZN,2025-12-31 18:59:59-05:00,2025-12-31,2026-01-02,2,2,200.0,250.0
1,86644,what-will-nflx-close-at-in-2025,What will Netflix (NFLX) close at in 2025?,NFLX,2025-12-31 18:59:59-05:00,2025-12-31,2026-01-02,2,2,85.0,125.0
2,89531,what-will-tesla-tsla-close-at-in-2025,What will Tesla (TSLA) close at in 2025?,TSLA,2025-12-30 19:00:00-05:00,2025-12-30,2026-01-02,3,3,300.0,500.0
3,89532,what-will-google-googl-close-at-in-2025,What will Google (GOOGL) close at in 2025?,GOOGL,2025-12-30 19:00:00-05:00,2025-12-30,2026-01-02,3,3,250.0,400.0
4,89541,what-will-nvidia-nvda-close-at-in-2025,What will NVIDIA (NVDA) close at in 2025?,NVDA,2025-12-30 19:00:00-05:00,2025-12-30,2026-01-02,3,3,160.0,220.0


In [128]:
# --------------------------------------------------
# Step 3.4: broader extraction universe
# Keep events with abs_mismatch_days <= 1
# --------------------------------------------------
event_expiry_step3 = event_expiry_with_strikes.loc[
    event_expiry_with_strikes["abs_mismatch_days"] <= 1
].copy()

print("Events in Step 3 universe:", len(event_expiry_step3))
event_expiry_step3.head()

Events in Step 3 universe: 455


,event_id,slug,title,stock_ticker,endDate_et,pm_enddate,matched_option_expiry,mismatch_days,abs_mismatch_days,min_pm_strike,max_pm_strike
17,112365,aapl-above-on-december-26-2025,Will Apple (AAPL) finish week of December 22 a...,AAPL,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0,240.0,300.0
18,112367,aapl-week-december-26-2025,Apple (AAPL) closes week of Dec 22 at ___?,AAPL,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0,250.0,295.0
19,112373,msft-week-december-26-2025,Microsoft (MSFT) closes week of Dec 22 at ___?,MSFT,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0,440.0,530.0
20,112374,msft-above-on-december-26-2025,Will Microsoft (MSFT) finish week of December ...,MSFT,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0,420.0,540.0
21,112378,amzn-above-on-december-26-2025,Will Amazon (AMZN) finish week of December 22 ...,AMZN,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0,195.0,255.0


In [129]:
# --------------------------------------------------
# Step 3.5: aggregate to stock × matched_option_expiry
# --------------------------------------------------
stock_expiry_needed = (
    event_expiry_step3.groupby(["stock_ticker", "matched_option_expiry"])
    .agg(
        n_events=("event_id", "size"),
        min_pm_strike=("min_pm_strike", "min"),
        max_pm_strike=("max_pm_strike", "max"),
    )
    .reset_index()
    .merge(stock_strike_steps_df, on="stock_ticker", how="left")
    .sort_values(["stock_ticker", "matched_option_expiry"])
    .reset_index(drop=True)
)

stock_expiry_needed.head(20)

,stock_ticker,matched_option_expiry,n_events,min_pm_strike,max_pm_strike,strike_step
0,AAPL,2025-12-26,2,240.0,300.0,2.5
1,AAPL,2026-01-02,2,245.0,305.0,2.5
2,AAPL,2026-01-09,2,240.0,300.0,2.5
3,AAPL,2026-01-16,2,230.0,290.0,2.5
4,AAPL,2026-01-23,3,230.0,290.0,2.5
5,AAPL,2026-01-30,5,210.0,330.0,2.5
6,AAPL,2026-02-02,2,250.0,270.0,2.5
7,AAPL,2026-02-04,2,260.0,280.0,2.5
8,AAPL,2026-02-06,3,230.0,290.0,2.5
9,AAPL,2026-02-09,2,265.0,290.0,2.5


In [130]:
# --------------------------------------------------
# Step 3.6: helper to generate strike grid
# --------------------------------------------------
def generate_strike_grid(min_strike, max_strike, step):
    if pd.isna(min_strike) or pd.isna(max_strike) or pd.isna(step):
        return []

    start = np.floor(min_strike / step) * step
    end = np.ceil(max_strike / step) * step

    n = int(round((end - start) / step))
    return [round(start + i * step, 10) for i in range(n + 1)]

In [131]:
# --------------------------------------------------
# Step 3.7: expand to stock × option expiry × strike
# --------------------------------------------------
rows = []

for _, r in stock_expiry_needed.iterrows():
    stock = r["stock_ticker"]
    expiry = r["matched_option_expiry"]
    min_k = float(r["min_pm_strike"])
    max_k = float(r["max_pm_strike"])
    step = float(r["strike_step"])

    for k in generate_strike_grid(min_k, max_k, step):
        rows.append({
            "stock_ticker": stock,
            "option_expiry": expiry,
            "strike": k,
            "strike_step": step,
        })

option_universe_step3 = (
    pd.DataFrame(rows)
    .sort_values(["stock_ticker", "option_expiry", "strike"])
    .reset_index(drop=True)
)

print(option_universe_step3.shape)
option_universe_step3.head(30)

(7091, 4)


,stock_ticker,option_expiry,strike,strike_step
0,AAPL,2025-12-26,240.0,2.5
1,AAPL,2025-12-26,242.5,2.5
2,AAPL,2025-12-26,245.0,2.5
3,AAPL,2025-12-26,247.5,2.5
4,AAPL,2025-12-26,250.0,2.5
5,AAPL,2025-12-26,252.5,2.5
6,AAPL,2025-12-26,255.0,2.5
7,AAPL,2025-12-26,257.5,2.5
8,AAPL,2025-12-26,260.0,2.5
9,AAPL,2025-12-26,262.5,2.5


In [132]:
# --------------------------------------------------
# Step 3.8: save
# --------------------------------------------------
option_universe_step3.to_csv("option_universe_step3_stock_expiry_strike.csv", index=False)
stock_expiry_needed.to_csv("stock_expiry_needed_step3.csv", index=False)

print("Saved:")
print("- option_universe_step3_stock_expiry_strike.csv")
print("- stock_expiry_needed_step3.csv")

Saved:
- option_universe_step3_stock_expiry_strike.csv
- stock_expiry_needed_step3.csv


In [134]:
pd.read_csv("option_universe_step3_stock_expiry_strike_result.csv")

,stock_ticker,option_expiry,strike,option_ticker,Option Expire Date,Expire Date Exist
0,AAPL,12/26/2025,240.0,AAPL US 12/26/25 C240 Equity,12/26/2025,True
1,AAPL,12/26/2025,242.5,AAPL US 12/26/25 C242.5 Equity,12/26/2025,True
2,AAPL,12/26/2025,245.0,AAPL US 12/26/25 C245 Equity,12/26/2025,True
3,AAPL,12/26/2025,247.5,AAPL US 12/26/25 C247.5 Equity,12/26/2025,True
4,AAPL,12/26/2025,250.0,AAPL US 12/26/25 C250 Equity,12/26/2025,True
...,...,...,...,...,...,...
7086,TSLA,4/2/2026,390.0,TSLA US 04/02/26 C390 Equity,4/2/2026,True
7087,TSLA,4/2/2026,392.5,TSLA US 04/02/26 C392.5 Equity,4/2/2026,True
7088,TSLA,4/2/2026,395.0,TSLA US 04/02/26 C395 Equity,4/2/2026,True
7089,TSLA,4/2/2026,397.5,TSLA US 04/02/26 C397.5 Equity,4/2/2026,True


## Filter option universe for pricing

We now reduce the option universe in two ways:

1. keep only events whose PM end date exactly matches the assigned option expiry,
2. for each PM boundary strike, keep only the locally relevant option strikes:
   - exact strike if available,
   - nearest smaller strike,
   - nearest larger strike.

This produces a much smaller list of option contracts that are directly useful for pricing.

In [154]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Read Step 3 Bloomberg existence result
# --------------------------------------------------
opt_exist = pd.read_csv("option_universe_step3_stock_expiry_strike_result.csv")

print(opt_exist.shape)
print(opt_exist.columns.tolist())
opt_exist.head()

(7091, 6)
['stock_ticker', 'option_expiry', 'strike', 'option_ticker', 'Option Expire Date', 'Expire Date Exist']


,stock_ticker,option_expiry,strike,option_ticker,Option Expire Date,Expire Date Exist
0,AAPL,12/26/2025,240.0,AAPL US 12/26/25 C240 Equity,12/26/2025,True
1,AAPL,12/26/2025,242.5,AAPL US 12/26/25 C242.5 Equity,12/26/2025,True
2,AAPL,12/26/2025,245.0,AAPL US 12/26/25 C245 Equity,12/26/2025,True
3,AAPL,12/26/2025,247.5,AAPL US 12/26/25 C247.5 Equity,12/26/2025,True
4,AAPL,12/26/2025,250.0,AAPL US 12/26/25 C250 Equity,12/26/2025,True


In [155]:
# --------------------------------------------------
# 2. Clean the option existence file
# --------------------------------------------------
opt_exist = opt_exist.copy()

opt_exist["option_expiry"] = pd.to_datetime(opt_exist["option_expiry"], errors="coerce").dt.date
opt_exist["strike"] = pd.to_numeric(opt_exist["strike"], errors="coerce")

opt_exist["Expire Date Exist"] = (
    opt_exist["Expire Date Exist"]
    .astype(str)
    .str.strip()
    .str.upper()
)

opt_exist["option_exists"] = opt_exist["Expire Date Exist"].isin(["TRUE", "YES", "Y", "1"])

# Keep only options that exist
opt_exist_avail = (
    opt_exist.loc[opt_exist["option_exists"]]
    .dropna(subset=["stock_ticker", "option_expiry", "strike"])
    .copy()
)

print("Available option rows:", len(opt_exist_avail))
opt_exist_avail.head()

Available option rows: 5584


,stock_ticker,option_expiry,strike,option_ticker,Option Expire Date,Expire Date Exist,option_exists
0,AAPL,2025-12-26,240.0,AAPL US 12/26/25 C240 Equity,12/26/2025,TRUE,True
1,AAPL,2025-12-26,242.5,AAPL US 12/26/25 C242.5 Equity,12/26/2025,TRUE,True
2,AAPL,2025-12-26,245.0,AAPL US 12/26/25 C245 Equity,12/26/2025,TRUE,True
3,AAPL,2025-12-26,247.5,AAPL US 12/26/25 C247.5 Equity,12/26/2025,TRUE,True
4,AAPL,2025-12-26,250.0,AAPL US 12/26/25 C250 Equity,12/26/2025,TRUE,True


In [156]:
# --------------------------------------------------
# 3. Keep only exact-expiry-match events
# --------------------------------------------------
# event_expiry_match came from Step 2 and already contains mismatch_days
event_exact = event_expiry_match.loc[event_expiry_match["mismatch_days"] == 0].copy()

print("Exact-match events:", len(event_exact))
event_exact.head()

Exact-match events: 355


,event_id,slug,title,stock_ticker,endDate_et,pm_enddate,matched_option_expiry,mismatch_days,abs_mismatch_days
144,112365,aapl-above-on-december-26-2025,Will Apple (AAPL) finish week of December 22 a...,AAPL,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0
145,112367,aapl-week-december-26-2025,Apple (AAPL) closes week of Dec 22 at ___?,AAPL,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0
146,112373,msft-week-december-26-2025,Microsoft (MSFT) closes week of Dec 22 at ___?,MSFT,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0
147,112374,msft-above-on-december-26-2025,Will Microsoft (MSFT) finish week of December ...,MSFT,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0
148,112378,amzn-above-on-december-26-2025,Will Amazon (AMZN) finish week of December 22 ...,AMZN,2025-12-26 16:00:00-05:00,2025-12-26,2025-12-26,0,0


In [157]:
# --------------------------------------------------
# 4. Build event-level PM boundary list
#    Use all finite positive PM boundaries from both lower and upper bounds
# --------------------------------------------------
event_boundaries_long = pd.concat([
    markets_data[["event_id", "lower_bound"]].rename(columns={"lower_bound": "boundary_strike"}),
    markets_data[["event_id", "upper_bound"]].rename(columns={"upper_bound": "boundary_strike"}),
], ignore_index=True)

event_boundaries_long["boundary_strike"] = pd.to_numeric(event_boundaries_long["boundary_strike"], errors="coerce")
event_boundaries_long = event_boundaries_long[
    np.isfinite(event_boundaries_long["boundary_strike"]) & (event_boundaries_long["boundary_strike"] > 0)
].copy()

event_boundaries = (
    event_boundaries_long.groupby("event_id")["boundary_strike"]
    .agg(lambda s: sorted(pd.unique(s).tolist()))
    .reset_index(name="pm_boundary_strikes")
)

event_boundaries.head()

,event_id,pm_boundary_strikes
0,86643,"[200.0, 205.0, 210.0, 215.0, 220.0, 225.0, 230..."
1,86644,"[85.0, 90.0, 95.0, 100.0, 105.0, 110.0, 115.0,..."
2,89531,"[300.0, 325.0, 350.0, 375.0, 400.0, 425.0, 450..."
3,89532,"[250.0, 265.0, 280.0, 295.0, 310.0, 325.0, 340..."
4,89541,"[160.0, 165.0, 170.0, 175.0, 180.0, 185.0, 190..."


In [158]:
# --------------------------------------------------
# 5. Attach PM boundary list to exact-match events
# --------------------------------------------------
event_exact = (
    event_exact
    .merge(event_boundaries, on="event_id", how="left")
    .copy()
)

event_exact[["event_id", "stock_ticker", "pm_enddate", "matched_option_expiry", "pm_boundary_strikes"]].head()

,event_id,stock_ticker,pm_enddate,matched_option_expiry,pm_boundary_strikes
0,112365,AAPL,2025-12-26,2025-12-26,"[240.0, 245.0, 250.0, 255.0, 260.0, 265.0, 270..."
1,112367,AAPL,2025-12-26,2025-12-26,"[250.0, 255.0, 260.0, 265.0, 270.0, 275.0, 280..."
2,112373,MSFT,2025-12-26,2025-12-26,"[440.0, 450.0, 460.0, 470.0, 480.0, 490.0, 500..."
3,112374,MSFT,2025-12-26,2025-12-26,"[420.0, 430.0, 440.0, 450.0, 460.0, 470.0, 480..."
4,112378,AMZN,2025-12-26,2025-12-26,"[195.0, 200.0, 205.0, 210.0, 215.0, 220.0, 225..."


In [163]:
# --------------------------------------------------
# 6. For each PM boundary strike, keep only:
#    - nearest smaller available strike
#    - nearest larger available strike
#    AND require each side gap < 2 * option step
# --------------------------------------------------
def bracketing_option_strikes_for_boundary(stock, expiry, boundary, option_df, step):
    sub = option_df.loc[
        (option_df["stock_ticker"] == stock) &
        (option_df["option_expiry"] == expiry),
        ["stock_ticker", "option_expiry", "strike"]
    ].drop_duplicates().sort_values("strike")

    if sub.empty:
        return []

    strikes = sub["strike"].to_numpy()

    lower = strikes[strikes < boundary]
    upper = strikes[strikes > boundary]

    out = []

    if len(lower) > 0:
        lower_k = float(lower[-1])
        if (boundary - lower_k) < 2 * step:
            out.append(lower_k)

    if len(upper) > 0:
        upper_k = float(upper[0])
        if (upper_k - boundary) < 2 * step:
            out.append(upper_k)

    return out

In [164]:
# --------------------------------------------------
# 7. Build filtered option list needed for pricing
#    using only lower + upper bracketing strikes
#    with gap restriction: each side gap < 2 * option step
# --------------------------------------------------
rows = []

for _, r in event_exact.iterrows():
    event_id = r["event_id"]
    stock = r["stock_ticker"]
    expiry = r["matched_option_expiry"]
    pm_end = r["pm_enddate"]
    slug = r["slug"]
    boundaries = r["pm_boundary_strikes"]

    step = stock_strike_steps[stock]

    if not isinstance(boundaries, list):
        continue

    for b in boundaries:
        needed_strikes = bracketing_option_strikes_for_boundary(
            stock, expiry, b, opt_exist_avail, step
        )

        for k in needed_strikes:
            rows.append({
                "event_id": event_id,
                "slug": slug,
                "stock_ticker": stock,
                "pm_enddate": pm_end,
                "option_expiry": expiry,
                "pm_boundary_strike": b,
                "selected_option_strike": k,
                "strike_relation": "lower" if k < b else "upper",
                "option_step": step,
                "strike_gap": abs(k - b),
            })

options_needed_for_pricing = (
    pd.DataFrame(rows)
    .drop_duplicates()
    .sort_values([
        "stock_ticker", "option_expiry", "event_id",
        "pm_boundary_strike", "selected_option_strike"
    ])
    .reset_index(drop=True)
)

print("Rows in options_needed_for_pricing:", len(options_needed_for_pricing))
options_needed_for_pricing.head(30)

Rows in options_needed_for_pricing: 5139


,event_id,slug,stock_ticker,pm_enddate,option_expiry,pm_boundary_strike,selected_option_strike,strike_relation,option_step,strike_gap
0,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,240.0,242.5,upper,2.5,2.5
1,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,245.0,242.5,lower,2.5,2.5
2,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,245.0,247.5,upper,2.5,2.5
3,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,250.0,247.5,lower,2.5,2.5
4,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,250.0,252.5,upper,2.5,2.5
5,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,255.0,252.5,lower,2.5,2.5
6,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,255.0,257.5,upper,2.5,2.5
7,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,260.0,257.5,lower,2.5,2.5
8,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,260.0,262.5,upper,2.5,2.5
9,112365,aapl-above-on-december-26-2025,AAPL,2025-12-26,2025-12-26,265.0,262.5,lower,2.5,2.5


In [165]:
# --------------------------------------------------
# 8. Collapse to unique option contracts actually needed
# --------------------------------------------------
unique_option_contracts_needed = (
    options_needed_for_pricing[[
        "stock_ticker", "option_expiry", "selected_option_strike"
    ]]
    .drop_duplicates()
    .rename(columns={"selected_option_strike": "strike"})
    .sort_values(["stock_ticker", "option_expiry", "strike"])
    .reset_index(drop=True)
)

print("Unique option contracts needed:", len(unique_option_contracts_needed))
unique_option_contracts_needed.head(30)

Unique option contracts needed: 1970


,stock_ticker,option_expiry,strike
0,AAPL,2025-12-26,242.5
1,AAPL,2025-12-26,247.5
2,AAPL,2025-12-26,252.5
3,AAPL,2025-12-26,257.5
4,AAPL,2025-12-26,262.5
5,AAPL,2025-12-26,267.5
6,AAPL,2025-12-26,272.5
7,AAPL,2025-12-26,277.5
8,AAPL,2025-12-26,282.5
9,AAPL,2025-12-26,287.5


In [166]:
# --------------------------------------------------
# 9. Summary counts
# --------------------------------------------------
print("Exact-match events retained:", len(event_exact))
print("Rows in event-boundary-to-option map:", len(options_needed_for_pricing))
print("Unique option contracts needed:", len(unique_option_contracts_needed))

summary_by_stock = (
    unique_option_contracts_needed.groupby("stock_ticker")
    .size()
    .reset_index(name="n_option_contracts")
    .sort_values("n_option_contracts", ascending=False)
    .reset_index(drop=True)
)

summary_by_stock

Exact-match events retained: 355
Rows in event-boundary-to-option map: 5139
Unique option contracts needed: 1970


,stock_ticker,n_option_contracts
0,META,345
1,MSFT,303
2,TSLA,242
3,NVDA,197
4,OPEN,190
5,AAPL,184
6,AMZN,177
7,GOOGL,169
8,NFLX,129
9,PLTR,34


In [168]:
# unique_option_contracts_needed is the filtered universe after:
# - exact expiry match only
# - bracketing strikes only
# - strike gap < 2 * option step

stock_expiry_groups = (
    unique_option_contracts_needed
    .groupby(["stock_ticker", "option_expiry"])
    .agg(
        n_option_contracts=("strike", "size"),
        min_strike=("strike", "min"),
        max_strike=("strike", "max"),
    )
    .reset_index()
    .sort_values(["stock_ticker", "option_expiry"])
    .reset_index(drop=True)
)

print("Number of unique stock-expiry groups:", len(stock_expiry_groups))
stock_expiry_groups.head(30)

Number of unique stock-expiry groups: 179


,stock_ticker,option_expiry,n_option_contracts,min_strike,max_strike
0,AAPL,2025-12-26,12,242.5,297.5
1,AAPL,2026-01-02,12,247.5,302.5
2,AAPL,2026-01-09,12,242.5,297.5
3,AAPL,2026-01-16,12,232.5,287.5
4,AAPL,2026-01-23,12,232.5,287.5
5,AAPL,2026-01-30,14,217.5,282.5
6,AAPL,2026-02-02,4,252.5,267.5
7,AAPL,2026-02-04,4,262.5,277.5
8,AAPL,2026-02-06,12,232.5,287.5
9,AAPL,2026-02-09,5,267.5,287.5


In [169]:
# Summary of how many groups and contracts by stock
group_summary_by_stock = (
    stock_expiry_groups.groupby("stock_ticker")
    .agg(
        n_expiry_groups=("option_expiry", "size"),
        total_option_contracts=("n_option_contracts", "sum"),
        avg_contracts_per_group=("n_option_contracts", "mean"),
        median_contracts_per_group=("n_option_contracts", "median"),
        max_contracts_in_group=("n_option_contracts", "max"),
    )
    .reset_index()
    .sort_values("total_option_contracts", ascending=False)
    .reset_index(drop=True)
)

group_summary_by_stock

,stock_ticker,n_expiry_groups,total_option_contracts,avg_contracts_per_group,median_contracts_per_group,max_contracts_in_group
0,META,20,345,17.250000,19.0,35
1,MSFT,20,303,15.150000,14.5,28
2,TSLA,21,242,11.523810,12.0,24
3,NVDA,24,197,8.208333,10.0,13
4,OPEN,12,190,15.833333,16.5,18
5,AAPL,22,184,8.363636,8.5,15
6,AMZN,21,177,8.428571,11.0,14
7,GOOGL,18,169,9.388889,12.0,16
8,NFLX,12,129,10.750000,8.0,17
9,PLTR,9,34,3.777778,3.0,9


In [170]:
# Events left: exact expiry match only
events_exact = event_expiry_match.loc[event_expiry_match["mismatch_days"] == 0].copy()

n_events_exact = len(events_exact)

# Markets left: all markets belonging to those exact-match events
exact_event_ids = set(events_exact["event_id"].astype(str))

markets_exact = markets_data.loc[
    markets_data["event_id"].astype(str).isin(exact_event_ids)
].copy()

n_markets_exact = len(markets_exact)

print("Exact-match events left :", n_events_exact)
print("Exact-match markets left:", n_markets_exact)

Exact-match events left : 355
Exact-match markets left: 3370


## Export selected option contracts for Excel ticker construction

We export the filtered option universe to a clean file for Excel.

Each row is one selected option contract, with:
- `stock_ticker`
- `option_expiry`
- `strike`
- `cp`

We keep `cp = C` for now, since the current pricing implementation uses calls only.

Bloomberg ticker will be constructed later in Excel.

In [172]:
import pandas as pd

# --------------------------------------------------
# Build export table for Excel
# --------------------------------------------------
options_for_excel = unique_option_contracts_needed.copy()

# calls only for now
options_for_excel["cp"] = "C"

# reorder columns
options_for_excel = options_for_excel[
    ["stock_ticker", "option_expiry", "strike", "cp"]
].sort_values(
    ["stock_ticker", "option_expiry", "strike"]
).reset_index(drop=True)

print(options_for_excel.shape)
options_for_excel.head(30)

(1970, 4)


,stock_ticker,option_expiry,strike,cp
0,AAPL,2025-12-26,242.5,C
1,AAPL,2025-12-26,247.5,C
2,AAPL,2025-12-26,252.5,C
3,AAPL,2025-12-26,257.5,C
4,AAPL,2025-12-26,262.5,C
5,AAPL,2025-12-26,267.5,C
6,AAPL,2025-12-26,272.5,C
7,AAPL,2025-12-26,277.5,C
8,AAPL,2025-12-26,282.5,C
9,AAPL,2025-12-26,287.5,C


In [173]:
#options_for_excel.to_csv("selected_options_for_excel.csv", index=False)
options_for_excel.to_excel("selected_options_for_excel.xlsx", index=False)

print("Saved:")
#print("- selected_options_for_excel.csv")
print("- selected_options_for_excel.xlsx")

Saved:
- selected_options_for_excel.xlsx


## Batch selected options by expiry date for Excel download

To simplify Bloomberg Excel extraction, we batch the selected option contracts by expiry date.

Current batches:

1. expiry on or before 2026-01-31
2. expiry in February 2026
3. expiry on or after 2026-03-01

Within each batch, we assign:
- a common start date for extraction,
- a common end date equal to the latest expiry in that batch.

This makes it easier to pull historical data in Excel with one date range per batch.

In [185]:
import pandas as pd
import numpy as np

# Start from your current filtered option universe
options_for_batching = unique_option_contracts_needed.copy()
options_for_batching["option_expiry"] = pd.to_datetime(
    options_for_batching["option_expiry"], errors="coerce"
)
options_for_batching["cp"] = "C"

In [186]:
# --------------------------------------------------
# Define finer batch labels
# --------------------------------------------------
def assign_expiry_batch(expiry):
    if pd.isna(expiry):
        return np.nan
    if expiry <= pd.Timestamp("2026-01-16"):
        return "batch_1a_to_2026_01_16"
    elif expiry <= pd.Timestamp("2026-01-31"):
        return "batch_1b_2026_01_17_to_2026_01_31"
    elif expiry <= pd.Timestamp("2026-02-13"):
        return "batch_2a_2026_02_01_to_2026_02_13"
    elif expiry <= pd.Timestamp("2026-02-28"):
        return "batch_2b_2026_02_14_to_2026_02_28"
    else:
        return "batch_3_from_2026_03_01"

options_for_batching["expiry_batch"] = options_for_batching["option_expiry"].apply(assign_expiry_batch)

options_for_batching["expiry_batch"].value_counts(dropna=False)

expiry_batch
batch_1a_to_2026_01_16               515
batch_2a_2026_02_01_to_2026_02_13    428
batch_2b_2026_02_14_to_2026_02_28    397
batch_3_from_2026_03_01              323
batch_1b_2026_01_17_to_2026_01_31    307
Name: count, dtype: int64

In [187]:
# --------------------------------------------------
# Assign common extraction dates by batch
# --------------------------------------------------
batch_end_dates = (
    options_for_batching.groupby("expiry_batch")["option_expiry"]
    .max()
    .to_dict()
)

options_for_batching["extract_start_date"] = pd.Timestamp("2025-12-20")
options_for_batching["extract_end_date"] = options_for_batching["expiry_batch"].map(batch_end_dates)

options_for_batching.head(20)

,stock_ticker,option_expiry,strike,cp,expiry_batch,extract_start_date,extract_end_date
0,AAPL,2025-12-26,242.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16
1,AAPL,2025-12-26,247.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16
2,AAPL,2025-12-26,252.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16
3,AAPL,2025-12-26,257.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16
4,AAPL,2025-12-26,262.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16
5,AAPL,2025-12-26,267.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16
6,AAPL,2025-12-26,272.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16
7,AAPL,2025-12-26,277.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16
8,AAPL,2025-12-26,282.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16
9,AAPL,2025-12-26,287.5,C,batch_1a_to_2026_01_16,2025-12-20,2026-01-16


In [188]:
# --------------------------------------------------
# Optional helper text columns for Excel
# --------------------------------------------------
options_for_batching["option_expiry_text"] = options_for_batching["option_expiry"].dt.strftime("%Y-%m-%d")
options_for_batching["extract_start_date_text"] = options_for_batching["extract_start_date"].dt.strftime("%Y%m%d")
options_for_batching["extract_end_date_text"] = options_for_batching["extract_end_date"].dt.strftime("%Y%m%d")

options_for_batching["cp"] = "C"

options_for_batching = options_for_batching[
    [
        "expiry_batch",
        "stock_ticker",
        "option_expiry",
        "option_expiry_text",
        "strike",
        "cp",
        "extract_start_date",
        "extract_end_date",
        "extract_start_date_text",
        "extract_end_date_text",
    ]
].sort_values(
    ["expiry_batch", "stock_ticker", "option_expiry", "strike"]
).reset_index(drop=True)

options_for_batching.head(30)

,expiry_batch,stock_ticker,option_expiry,option_expiry_text,strike,cp,extract_start_date,extract_end_date,extract_start_date_text,extract_end_date_text
0,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,242.5,C,2025-12-20,2026-01-16,20251220,20260116
1,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,247.5,C,2025-12-20,2026-01-16,20251220,20260116
2,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,252.5,C,2025-12-20,2026-01-16,20251220,20260116
3,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,257.5,C,2025-12-20,2026-01-16,20251220,20260116
4,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,262.5,C,2025-12-20,2026-01-16,20251220,20260116
5,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,267.5,C,2025-12-20,2026-01-16,20251220,20260116
6,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,272.5,C,2025-12-20,2026-01-16,20251220,20260116
7,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,277.5,C,2025-12-20,2026-01-16,20251220,20260116
8,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,282.5,C,2025-12-20,2026-01-16,20251220,20260116
9,batch_1a_to_2026_01_16,AAPL,2025-12-26,2025-12-26,287.5,C,2025-12-20,2026-01-16,20251220,20260116


In [189]:
# --------------------------------------------------
# Batch summaries
# --------------------------------------------------
batch_summary = (
    options_for_batching.groupby("expiry_batch")
    .agg(
        n_options=("strike", "size"),
        n_stocks=("stock_ticker", "nunique"),
        min_expiry=("option_expiry", "min"),
        max_expiry=("option_expiry", "max"),
    )
    .reset_index()
)

batch_summary

,expiry_batch,n_options,n_stocks,min_expiry,max_expiry
0,batch_1a_to_2026_01_16,515,10,2025-12-26,2026-01-16
1,batch_1b_2026_01_17_to_2026_01_31,307,10,2026-01-23,2026-01-30
2,batch_2a_2026_02_01_to_2026_02_13,428,10,2026-02-02,2026-02-13
3,batch_2b_2026_02_14_to_2026_02_28,397,10,2026-02-18,2026-02-27
4,batch_3_from_2026_03_01,323,10,2026-03-02,2026-04-06


In [190]:
# --------------------------------------------------
# Split into separate dataframes
# --------------------------------------------------
batch_1a = options_for_batching.loc[
    options_for_batching["expiry_batch"] == "batch_1a_to_2026_01_16"
].copy()

batch_1b = options_for_batching.loc[
    options_for_batching["expiry_batch"] == "batch_1b_2026_01_17_to_2026_01_31"
].copy()

batch_2a = options_for_batching.loc[
    options_for_batching["expiry_batch"] == "batch_2a_2026_02_01_to_2026_02_13"
].copy()

batch_2b = options_for_batching.loc[
    options_for_batching["expiry_batch"] == "batch_2b_2026_02_14_to_2026_02_28"
].copy()

batch_3 = options_for_batching.loc[
    options_for_batching["expiry_batch"] == "batch_3_from_2026_03_01"
].copy()

print("Batch 1A:", batch_1a.shape)
print("Batch 1B:", batch_1b.shape)
print("Batch 2A:", batch_2a.shape)
print("Batch 2B:", batch_2b.shape)
print("Batch 3 :", batch_3.shape)

Batch 1A: (515, 10)
Batch 1B: (307, 10)
Batch 2A: (428, 10)
Batch 2B: (397, 10)
Batch 3 : (323, 10)


In [191]:
# --------------------------------------------------
# Save batch files
# --------------------------------------------------
options_for_batching.to_csv("selected_options_batched_all.csv", index=False)
batch_summary.to_csv("selected_options_batch_summary.csv", index=False)

batch_1a.to_csv("selected_options_batch_1a_to_2026_01_16.csv", index=False)
batch_1b.to_csv("selected_options_batch_1b_2026_01_17_to_2026_01_31.csv", index=False)
batch_2a.to_csv("selected_options_batch_2a_2026_02_01_to_2026_02_13.csv", index=False)
batch_2b.to_csv("selected_options_batch_2b_2026_02_14_to_2026_02_28.csv", index=False)
batch_3.to_csv("selected_options_batch_3_from_2026_03_01.csv", index=False)

print("Saved:")
print("- selected_options_batched_all.csv")
print("- selected_options_batch_summary.csv")
print("- selected_options_batch_1a_to_2026_01_16.csv")
print("- selected_options_batch_1b_2026_01_17_to_2026_01_31.csv")
print("- selected_options_batch_2a_2026_02_01_to_2026_02_13.csv")
print("- selected_options_batch_2b_2026_02_14_to_2026_02_28.csv")
print("- selected_options_batch_3_from_2026_03_01.csv")

Saved:
- selected_options_batched_all.csv
- selected_options_batch_summary.csv
- selected_options_batch_1a_to_2026_01_16.csv
- selected_options_batch_1b_2026_01_17_to_2026_01_31.csv
- selected_options_batch_2a_2026_02_01_to_2026_02_13.csv
- selected_options_batch_2b_2026_02_14_to_2026_02_28.csv
- selected_options_batch_3_from_2026_03_01.csv
